<!--HEADER-->
(điền sau khi chạy)

## Phân công nhóm

| Mã | Việc | Đầu ra | Mục trong notebook |
|---|---|---|---|
| **T1** ✔ | Dữ liệu 2020–2025 (vnstock/nguồn trả phí) + **mã đã hủy niêm yết** để loại survivorship bias — *đã làm: HOSE 2015–11/2026* | `prices.csv`, `values.csv` | §2, ô T1 |
| **T2** ✔ | Độ vững: chi phí GD thực tế (phí + thuế 0,1%), tần suất tái cân bằng 5/21/63 phiên, xử lý phiên không giao dịch (~18%) | Bảng độ vững | §6, ô T2 |
| **T3** ✔ | Đối chứng & kiểm định chuẩn: danh mục 1/N, mô hình nhân tố, DCC-GARCH; kiểm định chênh lệch phương sai Ledoit–Wolf (2011, HAC), Sharpe (Jobson–Korkie/Memmel) | Bảng kiểm định | §6, ô T3 |
| **T4** (để sau) | Lý thuyết: chứng minh tính đúng của việc chọn nhánh, điều kiện ngoại lai, hiệu chỉnh N hữu hạn; **rà soát tính mới** (đọc toàn văn BBP 2016 §III và Phys. Rep. 2017) | Mục Methodology (LaTeX) | — |
| **T5** ✔ | Vì sao VS giúp trên HOSE nhưng không trong GARCH? (biến động riêng lẻ, tương quan đổi theo chế độ…) | Ghi chú + thí nghiệm | §6, ô T5 |
| **T6** ✔ | Mở rộng: N = 50/150, cửa sổ T, VN30/HNX | Bảng phụ lục | §5 (đổi CONFIG) |
| **T7** | Viết bài, nhắm tạp chí (kiểm tra hạng Q trên Scimago) | Bản thảo | — |

**Quy ước:** mỗi người làm trên bản sao notebook (`File → Save a copy in Drive`), ghi kết quả vào thư mục chung. Đổi tham số ở ô CONFIG, không sửa thư viện ở §1 trừ khi đã thống nhất.

**Bản này khác file của thầy ở đâu.** Các ô của thầy (§1 thư viện, §1b kiểm thử, §3 backtest, §4 chạy chính, §5, khung T2/T3/T5) **giữ nguyên**. Chỉ đổi: ô CONFIG (thêm `ZERO_MAX`, `DATA_URL`, `DATA_DIR`, `DATA_SOURCE = "csv"`), đường dẫn đọc dữ liệu ở §2, và cờ `RUN_SENS = True` ở §5. Mọi ô thêm vào có dấu **▶**:
- **Dữ liệu** HOSE 2015 → nay, có mã hủy niêm yết, tải từ GitHub (`data_hose/`).
- **Hai bổ sung theo gợi ý của thầy:** sửa EW-NL-LW khi N > T_eff; thêm đối chứng EW-WeSpeR (Oriol 2024).
- **Lọc mã đứng giá** `ZERO_MAX = 20%` rồi chạy lại toàn bộ.
- Điền **T1, T2, T3, T5, T6**. T4 (lý thuyết) để sau.

## 0. Cài đặt & cấu hình

In [ ]:
import os, sys, subprocess, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB, "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# ======================= CONFIG =======================
CONFIG = dict(
    N=100,            # số mã thanh khoản nhất mỗi kỳ
    WINDOW=250,       # cửa sổ ước lượng (phiên)
    REB=21,           # tái cân bằng mỗi REB phiên
    COST=0.0025,      # chi phí 1 chiều (giả định, T2 cần chỉnh)
    HL_EW=60,         # half-life trọng số mũ
    HL_VOL=20,        # half-life chuẩn hóa biến động (VS)
    MAX_ABS_RET=0.15, # |r| > ngưỡng coi là lỗi dữ liệu (biên độ HOSE ±7%)
    ZERO_MAX=0.20,    # ▶ loại mã đứng giá: tỷ lệ phiên r = 0 trong cửa sổ quá khứ > ZERO_MAX (1.0 = không lọc, như bản gốc)
    QUICK=False,      # True: chạy nhanh với ít bộ ước lượng
)
DATA_SOURCE = "csv"      # ▶ "csv": dữ liệu HOSE 2015–11/2026 (có mã hủy niêm yết) | "github" (2006–3/2020, cũ) | "vnstock"
DATA_URL = "https://raw.githubusercontent.com/chinguyenkhac/data_HOSE/main/data_hose"   # ▶ link dữ liệu T1 trên GitHub (repo public)
WORKDIR = os.environ.get("RMT_WORKDIR", "/content/rmt" if IN_COLAB else "./rmt_work")
DATA_DIR = f"{WORKDIR}/data_hose"      # ▶ nơi lưu dữ liệu đã tải
os.makedirs(WORKDIR, exist_ok=True)
CONFIG

### ▶ Giao diện hình dùng chung
Một bảng màu cố định cho mỗi phương pháp (EW-RIE* luôn xanh lá, NL-LW xanh dương, EW-NL-LW cam, EW-WeSpeR tím, VS-NL-LW vàng) và kiểu hình thống nhất cho mọi hình bên dưới.

In [ ]:
# ▶ Giao diện hình dùng chung: một bảng màu cố định cho mỗi phương pháp, nền sáng, lưới nhạt, bỏ khung trên/phải
from matplotlib.colors import TwoSlopeNorm
COL = {"NL-LW": "#2a78d6", "EW-NL-LW": "#eb6834", "EW-RIE*": "#1baf7a", "EW-WeSpeR": "#8e44ad",
       "VS-NL-LW": "#eda100", "VS-RIE": "#b8860b", "1/N": "#9aa0a6", "PCA-auto": "#5d6d7e", "DCC-GARCH(1,1)": "#34495e"}
GREY = "#b8bcc2"
color_of = lambda name: COL.get(name, GREY)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "figure.facecolor": "white",
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "grid.color": "#e6e8eb", "grid.linewidth": .8,
    "legend.frameon": False, "legend.fontsize": 9})

## 1. Thư viện: các bộ ước lượng
Không cần sửa. Mỗi bộ ước lượng nhận ma trận lợi suất `X` (T×N) và trả về ma trận hiệp phương sai N×N.

In [ ]:
# ---------- tiện ích ----------
def cov_to_corr(S):
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def renorm(C):
    d = np.sqrt(np.diag(C)); return C / np.outer(d, d)

# ---------- đối chứng cổ điển ----------
def est_sample(X): return np.cov(X, rowvar=False)

def est_lw(X): return LedoitWolf().fit(X).covariance_

def nl_lw_S(S, n):
    # Ledoit & Wolf (2020), co phi tuyến giải tích; S: hiệp phương sai, n: cỡ mẫu (hiệu dụng)
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    L = np.tile(lam, (p, 1)).T; H = n ** (-1 / 3) * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    d = lam / ((np.pi*c*lam*ft)**2 + (1 - c - np.pi*c*lam*Ht)**2)
    return (U * d) @ U.T

def est_nllw(X):
    Xc = X - X.mean(0); n = len(X) - 1
    return nl_lw_S(Xc.T @ Xc / n, n)

def mp_corr(C, q):
    lam, V = np.linalg.eigh(C); N = len(C)
    lp = (1 - lam[-1] / N) * (1 + np.sqrt(q))**2
    noise = lam < lp; lam = lam.copy()
    if noise.any(): lam[noise] = lam[noise].mean()
    return renorm((V * lam) @ V.T)

def rie_corr(C, q):
    # RIE Bun–Bouchaud–Potters (2016) + hiệu chỉnh N hữu hạn (2017, §8). Đây là bản GỐC (đối chứng).
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10)
    z = lam - 1j / np.sqrt(N); g = (1 / (z[:, None] - lam[None, :])).mean(1)
    xi = lam / np.abs(1 - q + q*z*g)**2
    s2 = lam[0] / (1 - np.sqrt(q))**2 if q < 1 else lam.mean()
    lp, lm = s2 * (1 + np.sqrt(q))**2, s2 * (1 - np.sqrt(q))**2
    gmp = (z + s2*(q - 1) - np.sqrt(z - lm) * np.sqrt(z - lp)) / (2*q*z*s2)
    xi = xi * np.maximum(1, s2 * np.abs(1 - q + q*z*gmp)**2 / lam)
    xi *= N / xi.sum(); return renorm((V * xi) @ V.T)

def _corr_wrap(cleaner):
    def est(X):
        T, N = X.shape; C, sd = cov_to_corr(np.cov(X, rowvar=False))
        return cleaner(C, N / T) * np.outer(sd, sd)
    return est
est_mp, est_rie = _corr_wrap(mp_corr), _corr_wrap(rie_corr)

In [ ]:
# ---------- EW (trọng số mũ) và VS (chuẩn hóa biến động) ----------
def ew_weights(T, hl):
    w = 0.5 ** ((T - 1 - np.arange(T)) / hl); return w / w.sum()

def ew_corr(X, w):
    Xc = X - w @ X; S = (Xc * w[:, None]).T @ Xc
    sd = np.sqrt(np.diag(S)); return S / np.outer(sd, sd), sd

def make_ew(cleaner, hl):
    # cleaner(C, w) -> tương quan đã làm sạch
    def est(X):
        w = ew_weights(len(X), hl); C, sd = ew_corr(X, w)
        return cleaner(C, w) * np.outer(sd, sd)
    return est

def ewma_standardize(X, hl):
    lam = 0.5 ** (1 / hl); s2 = X[:hl].var(0) + 1e-10; Z = np.empty_like(X)
    for t in range(len(X)):
        Z[t] = X[t] / np.sqrt(s2); s2 = lam * s2 + (1 - lam) * X[t]**2
    return Z, np.sqrt(s2)

def make_vs(corr_cleaner, hl_vol):
    # corr_cleaner(Z) -> ma trận tương quan đã làm sạch của lợi suất chuẩn hóa Z
    def est(X):
        Z, sig = ewma_standardize(X, hl_vol); Z = Z[hl_vol:]
        return corr_cleaner(Z) * np.outer(sig, sig)
    return est

In [ ]:
# ---------- EW-RIE*: RIE chính xác cho trọng số mũ (phương pháp đề xuất) ----------
_STATS = {"fail": 0, "total": 0}; _NULL_CACHE = {}

def _F(g, t, a, w):
    d = 1 - a[None, :] * g[:, None]; R = (w[None, :] / d).sum(1)
    return g * R - t, R, (w[None, :] * a[None, :] / d**2).sum(1)

def _solve_gamma(t, a, w, g0, iters=30):
    # Newton có tìm kiếm lùi cho  γ·R_W(γ) = t  (vector phức), a = N·w
    g = g0.copy(); f, R, dR = _F(g, t, a, w)
    for _ in range(iters):
        step = f / (R + g * dR); lam_ = np.ones(len(g))
        for _ in range(30):
            gn = g - lam_ * step; fn, Rn, dRn = _F(gn, t, a, w)
            bad = ~(np.abs(fn) < np.abs(f)) & (np.abs(f) > 1e-14)
            if not bad.any(): break
            lam_[bad] *= 0.5
        keep = bad | ~np.isfinite(fn)
        gn[keep], fn[keep], Rn[keep], dRn[keep] = g[keep], f[keep], R[keep], dR[keep]
        g, f, R, dR = gn, fn, Rn, dRn
        if np.max(np.abs(f)) < 1e-13: break
    return g, np.abs(f)

def _continuation(t, a, w, steps=60, strict=True):
    # Nhánh chính: γ ≈ t khi t→0, đi theo tia s·t (s: 0→1)
    g = 1e-3 * t
    for s in np.geomspace(1e-3, 1.0, steps):
        g, res = _solve_gamma(s * t, a, w, g)
    ok = np.isfinite(g) & (res < 1e-8)
    if strict and not ok.all(): raise RuntimeError("Newton không hội tụ")
    return g if strict else (g, ok)

def _xi(spec, x, w, N, eta):
    z = x - 1j * eta; g = (1 / (z[:, None] - spec[None, :])).mean(1); t = z * g - 1
    gam, ok = _continuation(t, N * w, w, strict=False)
    gam[~ok] = (t / (1 + N * (w**2).sum() * t))[~ok]       # dự phòng (thực tế: 0 lần trên HOSE)
    _STATS["fail"] += int((~ok).sum()); _STATS["total"] += len(ok)
    return x * gam.imag / t.imag

def _xi_outliers(lam, idx, w, N):
    a = N * w; out = np.empty(len(idx))
    for k, i in enumerate(idx):
        g = np.sum(1 / (lam[i] - np.delete(lam, i))) / N     # Stieltjes bỏ-một trên trục thực
        t = np.array([lam[i] * g - 1 + 0j])
        try:
            gam = _continuation(t, a, w); _, R, dR = _F(gam, t, a, w)
            out[k] = lam[i] * np.real(1 / (R + gam * dR))[0]   # ξ = λ·γ'(t)
        except RuntimeError:
            out[k] = _xi(lam, lam[i:i+1], w, N, 1 / np.sqrt(N))[0]
    return out

def _null_spectrum(N, w, reps=8, seed=0):
    key = (N, len(w), round(float(w[-1]), 12))
    if key not in _NULL_CACHE:
        rng = np.random.default_rng(seed)
        ev = [np.linalg.eigvalsh((Y * w[:, None]).T @ Y) for Y in (rng.standard_normal((len(w), N)) for _ in range(reps))]
        _NULL_CACHE[key] = np.sort(np.concatenate(ev))
    return _NULL_CACHE[key]

def ew_rie_corr(C, w, debias=True):
    N = len(C); lam, V = np.linalg.eigh(C); lam = np.maximum(lam, 1e-10); eta = 1 / np.sqrt(N)
    null = _null_spectrum(N, w); s2 = np.median(lam) / np.median(null)
    D = lam[:, None] - lam[None, :]
    dens = (eta / (D**2 + eta**2)).sum(1) - 1 / eta            # mật độ lân cận (bỏ tự thân)
    out = np.where(dens < 0.1 / eta)[0]; bulk = np.setdiff1d(np.arange(N), out)
    xi = np.empty(N)
    xi[bulk] = _xi(lam, lam[bulk], w, N, eta)
    if len(out): xi[out] = _xi_outliers(lam, out, w, N)
    if debias and len(bulk):
        xi_null = _xi(null, lam[bulk] / s2, w, N, eta)
        xi[bulk] *= np.maximum(1, 1 / np.maximum(xi_null, 1e-6))
    xi = np.maximum(xi, 1e-8); xi *= N / xi.sum()
    return renorm((V * xi) @ V.T)

In [ ]:
# ---------- Danh sách bộ ước lượng ----------
def build_estimators(cfg):
    hl, hv = cfg["HL_EW"], cfg["HL_VOL"]
    E = {
        "Sample": est_sample,
        "Ledoit-Wolf": est_lw,
        "NL-LW": est_nllw,
        "MP-clip": est_mp,
        "RIE": est_rie,
        "VS-NL-LW": make_vs(lambda Z: renorm(nl_lw_S(np.cov(Z, rowvar=False), len(Z) - 1)), hv),
        "VS-RIE": make_vs(lambda Z: rie_corr(np.corrcoef(Z, rowvar=False), Z.shape[1] / len(Z)), hv),
        "EW-NL-LW": make_ew(lambda C, w: renorm(nl_lw_S(C, 1 / (w**2).sum())), hl),
        "EW-RIE (q_eff)": make_ew(lambda C, w: rie_corr(C, len(C) * (w**2).sum()), hl),
        "EW-RIE*": make_ew(ew_rie_corr, hl),
    }
    if cfg["QUICK"]:
        E = {k: E[k] for k in ["Sample", "NL-LW", "RIE", "EW-NL-LW", "EW-RIE*"]}
    return E

### ▶ §1c. Hai bổ sung theo gợi ý của thầy (ô mới, không sửa §1)
- **Sửa EW-NL-LW khi N > T_eff:** `nl_lw_S` được định nghĩa lại (nhánh p > n của Ledoit–Wolf 2020); bản của thầy giữ làm `nl_lw_S_v2`. Khi N ≤ T_eff kết quả trùng hệt.
- **Đối chứng EW-WeSpeR** (Oriol 2024, arXiv:2410.14413): khôi phục phổ quần thể của hiệp phương sai có trọng số rồi co phi tuyến (kiểu QuEST). Cài lại bằng numpy; `build_estimators` được bọc để thêm "EW-WeSpeR".

In [ ]:
import hashlib
nl_lw_S_v2 = nl_lw_S        # bản của thầy (chỉ đúng khi N ≤ T_eff), giữ để đối chứng

# ---------- v3: NL-LW cho cả trường hợp p > n (N > T_eff) ----------
def nl_lw_S(S, n):
    # Ledoit & Wolf (2020), co phi tuyến giải tích; S: hiệp phương sai, n: cỡ mẫu (hiệu dụng, có thể không nguyên).
    # p <= n: công thức gốc (giống hệt v2).  p > n: nhánh "p > n" trong mã gốc analytical_shrinkage.m của LW (2020):
    # chỉ ⌊n⌋ trị riêng lớn nhất được xem là "khác 0" (dùng cho mật độ nhân), các trị riêng còn lại nhận chung d̃₀
    # (d̃₀ dựa trên Hilbert transform tại 0). v2 dùng công thức p <= n với c = p/n > 1 ⇒ mẫu số (1 − c − πcλH)² sai, co sai các trị riêng nhỏ.
    p = len(S); lam, U = np.linalg.eigh(S); lam = np.maximum(lam, 1e-12)
    m = p if p <= n else max(int(np.floor(n)), 2)
    lz = lam[p - m:]; h = n ** (-1 / 3)
    L = np.tile(lz, (m, 1)).T; H = h * L.T; x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x**2 / 5, 0) / H, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Hf = (-3/10/np.pi) * x + (3/4/np.sqrt(5)/np.pi) * (1 - x**2/5) * np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    e = np.isclose(np.abs(x), np.sqrt(5)); Hf[e] = (-3/10/np.pi) * x[e]
    Ht = np.mean(Hf / H, axis=1); c = p / n
    if p <= n:
        d = lz / ((np.pi*c*lz*ft)**2 + (1 - c - np.pi*c*lz*Ht)**2)
    else:
        Hf0 = (1/np.pi) * (3/10/h**2 + 3/4/np.sqrt(5)/h * (1 - 1/5/h**2) * np.log((1 + np.sqrt(5)*h) / (1 - np.sqrt(5)*h))) * np.mean(1 / lz)
        d0 = 1 / (np.pi * (p - n) / n * Hf0)
        d1 = lz / (np.pi**2 * lz**2 * (ft**2 + Ht**2))
        d = np.concatenate([np.full(p - m, d0), d1])
    return (U * d) @ U.T

In [ ]:
# ---------- v3: EW-WeSpeR – đối chứng kiểu QuEST cho hiệp phương sai có trọng số (Oriol 2024, arXiv:2410.14413) ----------
# Mô hình: E = Σ_t w_t x_t x_tᵀ, x_t = C^{1/2} y_t, Σw = 1. Với r(e) = Σ_t w_t / (1 + N w_t e) (= R_W(−e)):
#   e(z) = (1/N) Σ_i τ_i / (τ_i r(e(z)) − z),      m(z) = (1/N) Σ_i 1 / (τ_i r(e(z)) − z)     (Im z > 0)
# m là Stieltjes của phổ mẫu tiệm cận (mật độ ρ = Im m / π); e = (1/N) tr C·G(z). Co phi tuyến "oracle":
#   d(λ) = lim u_λᵀ C u_λ = Im e(λ + i0) / Im m(λ + i0)      (trọng số đều ⇒ λ / |1 − q − qλ m|², Ledoit–Péché).
# WeSpeR = (1) tìm phổ quần thể τ sao cho phân vị của ρ_τ khớp trị riêng mẫu (như QuEST, nhưng với trọng số),
#          (2) áp d(λ_i) với τ đã khớp. Khác EW-RIE*: EW-RIE* dùng trực tiếp Stieltjes thực nghiệm (làm trơn η = N^{-1/2}),
#          WeSpeR dùng Stieltjes của MÔ HÌNH đã khớp ⇒ không cần η, không cần xử lý ngoại lai/hiệu chỉnh N hữu hạn riêng.
# Ghi chú: cài lại bằng numpy từ các phương trình trên (bản gốc của tác giả viết bằng PyTorch + autodiff; phiên làm việc này
# không truy cập được arXiv). τ tham số hóa: log τ tuyến tính từng khúc theo hạng (K nút) + các ngoại lai tự do;
# tối ưu bằng least_squares (Levenberg–Marquardt kiểu trf), sai số tương đối log λ̂_i − log λ_i; khởi tạo ấm từ kỳ trước.
_WSP = {"nfev": [], "cost": [], "warm": {}}

def _wsp_bins(w, nb=40):
    # nén phân phối trọng số thành nb nhóm (r chỉ phụ thuộc phân phối của w): giá trị TB và khối lượng Σw mỗi nhóm
    ws = np.sort(w); g = np.array_split(ws, min(nb, len(ws)))
    return np.array([x.mean() for x in g]), np.array([x.sum() for x in g])

def _wsp_solve(z, tau, wv, wm, N, e0=None, iters=40):
    # Newton (có tìm kiếm lùi, giữ Im e > 0) cho F(e) = e − mean τ/(τ r(e) − z), vector theo z
    def F(e):
        den = 1 + N * wv[None, :] * e[:, None]
        r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
        q = tau[None, :] * r[:, None] - z[:, None]
        f = e - (tau[None, :] / q).mean(1)
        df = 1 + (tau[None, :]**2 * dr[:, None] / q**2).mean(1)
        return f, df, q
    e = (tau[None, :] / (tau[None, :] - z[:, None])).mean(1) if e0 is None else e0.copy()
    f, df, q = F(e)
    for _ in range(iters):
        st = f / df; a = np.ones(len(e))
        for _ in range(25):
            en = e - a * st; fn, dfn, qn = F(en)
            bad = (~np.isfinite(fn)) | (np.abs(fn) > np.abs(f)) | (en.imag <= 0)
            bad &= np.abs(f) > 1e-13
            if not bad.any(): break
            a[bad] *= 0.5
        keep = bad
        en[keep], fn[keep], dfn[keep], qn[keep] = e[keep], f[keep], df[keep], q[keep]
        e, f, df, q = en, fn, dfn, qn
        if np.max(np.abs(f)) < 1e-11: break
    return e, (1 / q).mean(1), np.abs(f)

def _wsp_cold(x, eta, tau, wv, wm, N, steps=14):
    # tiếp tục theo η: từ η lớn (e ≈ mean τ/(τ − z)) xuống η đích ⇒ đúng nhánh Stieltjes (Im e > 0)
    e = None
    for s in np.geomspace(10 * max(x.max(), tau.max()), eta.min(), steps):
        e, m, res = _wsp_solve(x + 1j * np.maximum(eta, s), tau, wv, wm, N, e)
    return _wsp_solve(x + 1j * eta, tau, wv, wm, N, e)

def _wsp_quantiles(x, m, n_nz, N):
    # phân vị của phần khối lượng dương của ρ = Im m/π trên lưới x (hình thang), khớp với n_nz trị riêng khác 0
    rho = np.maximum(m.imag, 0) / np.pi; cdf = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * np.diff(x))])
    cdf /= cdf[-1]; u = (np.arange(n_nz) + 0.5) / n_nz
    return np.interp(u, cdf, x)

def _wsp_fwd(x, eta, tau, wv, wm, N, e0=None):
    # giải e trên lưới (ấm nếu có e0, nếu không/không hội tụ thì tiếp tục theo η) + các đạo hàm theo τ (đạo hàm ẩn)
    ok = False
    if e0 is not None:
        e, m, res = _wsp_solve(x + 1j * eta, tau, wv, wm, N, e0, iters=15); ok = np.isfinite(e).all() and res.max() < 1e-9
    if not ok: e, m, res = _wsp_cold(x, eta, tau, wv, wm, N)
    return e, m, res

def _wsp_jac_m(x, eta, e, tau, wv, wm, N):
    # dm/dτ_j (G×N):  F(e,τ) = e − mean τ/q, q = τ r(e) − z
    z = x + 1j * eta; den = 1 + N * wv[None, :] * e[:, None]
    r = (wm[None, :] / den).sum(1); dr = -(N * wm[None, :] * wv[None, :] / den**2).sum(1)
    q = tau[None, :] * r[:, None] - z[:, None]; q2 = q**2
    dF = 1 + (tau[None, :]**2 * dr[:, None] / q2).mean(1)
    de = -(z[:, None] / (N * q2)) / dF[:, None]
    A = -(tau[None, :] * dr[:, None] / q2).mean(1)
    return -(r[:, None] / (N * q2)) + A[:, None] * de

def ew_wesper_corr(C, w, K=8, max_nfev=25, G=350, return_info=False):
    from scipy.optimize import least_squares
    N = len(C); lam, V = np.linalg.eigh(C); T = len(w)
    n_nz = min(N, T - 1); lz = np.maximum(lam[N - n_nz:], 1e-10)          # trị riêng khác 0 (tăng dần)
    wv, wm = _wsp_bins(w)
    eta0 = 1 / np.sqrt(N); D = lz[:, None] - lz[None, :]                    # ngoại lai: như ew_rie_corr
    dens = (eta0 / (D**2 + eta0**2)).sum(1) - 1 / eta0
    n_out = int(min(np.sum(dens < 0.1 / eta0), 5)); nb = N - n_out
    knots = np.linspace(0, 1, K); rk = (np.arange(nb) + 0.5) / nb
    def tau_of(th):
        tb = np.exp(np.interp(rk, knots, np.cumsum(np.concatenate([[th[0]], np.exp(th[1:K])]))))
        return np.concatenate([tb, np.exp(th[K:])]) if n_out else tb
    x = np.unique(np.concatenate([np.geomspace(lz[0] * .25, lz[-1] * 1.6, G),
                                  *[lz[-k] * np.linspace(.6, 1.4, 60) for k in range(1, n_out + 1)]]))
    eta = 2e-3 * x; dx = np.diff(x); u = (np.arange(n_nz) + 0.5) / n_nz
    key = (N, T, K, n_out, round(float(w[-1]), 12))
    qb = np.maximum.accumulate(np.log(np.quantile(lz[:n_nz - n_out] if n_nz > n_out else lz, knots)))
    th_cold = np.concatenate([[qb[0]], np.log(np.maximum(np.diff(qb), 1e-3)), np.log(lz[n_nz - n_out:] * .9) if n_out else []])
    st = {}
    def evaluate(th):
        if st.get("th") is not None and np.array_equal(st["th"], th): return
        tau = tau_of(th); e, m, res = _wsp_fwd(x, eta, tau, wv, wm, N, st.get("e"))
        rho = np.maximum(m.imag, 0) / np.pi; raw = np.concatenate([[0], np.cumsum(0.5 * (rho[1:] + rho[:-1]) * dx)])
        tot = raw[-1]; cdf = raw / tot; qh = np.interp(u, cdf, x)
        st.update(th=th.copy(), tau=tau, e=e, m=m, rho=rho, raw=raw, tot=tot, cdf=cdf, qh=qh, ok=np.isfinite(qh).all())
    def fun(th):
        evaluate(th); return np.log(st["qh"]) - np.log(lz) if st["ok"] else np.full(n_nz, 10.0)
    def jac(th):
        evaluate(th)
        dm = _wsp_jac_m(x, eta, st["e"], st["tau"], wv, wm, N); dr = dm.imag / np.pi
        draw = np.vstack([np.zeros(N), np.cumsum(0.5 * (dr[1:] + dr[:-1]) * dx[:, None], axis=0)])
        dcdf = (draw - st["cdf"][:, None] * draw[-1][None, :]) / st["tot"]
        k = np.clip(np.searchsorted(x, st["qh"]) - 1, 0, len(x) - 2); f = (st["qh"] - x[k]) / dx[k]
        dC = dcdf[k] * (1 - f)[:, None] + dcdf[k + 1] * f[:, None]
        pdf = np.maximum((st["rho"][k] * (1 - f) + st["rho"][k + 1] * f) / st["tot"], 1e-3 / (lz[-1]))
        Jt = -dC / pdf[:, None] / st["qh"][:, None]                              # ∂ log λ̂ / ∂τ
        h = 1e-6; t0 = tau_of(th)
        Tt = np.column_stack([(tau_of(th + h * np.eye(len(th))[j]) - t0) / h for j in range(len(th))])
        return Jt @ Tt
    fit = lambda th0: (st.clear(), least_squares(fun, th0, jac=jac, method="trf", max_nfev=max_nfev, xtol=1e-4, ftol=1e-4))[1]
    sol = fit(_WSP["warm"].get(key, th_cold))                 # khởi tạo ấm từ kỳ trước (cùng N, T, h) nếu có
    if key in _WSP["warm"] and np.sqrt(np.mean(sol.fun**2)) > 0.05:  # ấm hỏng (cực tiểu địa phương) ⇒ khớp lại từ đầu, giữ nghiệm tốt hơn
        s2 = fit(th_cold); sol = s2 if np.sum(s2.fun**2) < np.sum(sol.fun**2) else sol
    _WSP["warm"][key] = sol.x; _WSP["nfev"].append(sol.nfev); _WSP["cost"].append(np.sqrt(np.mean(sol.fun**2)))
    tau = tau_of(sol.x)
    # co: d(λ_i) = Im e / Im m tại λ_i + iη (η nhỏ; ở khe/ngoài giá đỡ tỷ số → e'(λ)/m'(λ), vẫn xác định)
    ee, mm, _ = _wsp_cold(lz, 1e-4 * lz, tau, wv, wm, N); d = ee.imag / mm.imag
    xi = np.empty(N); xi[N - n_nz:] = np.maximum(d, 1e-8)
    if n_nz < N:   # không gian null (N ≥ T): giữ vết (tr C = N) cho các hướng null
        xi[:N - n_nz] = max((N - xi[N - n_nz:].sum()) / (N - n_nz), 1e-3 * np.mean(xi[N - n_nz:]))
    xi *= N / xi.sum(); out = renorm((V * xi) @ V.T)
    return (out, dict(tau=tau, xi=xi, lam=lam, rmse=_WSP["cost"][-1], nfev=sol.nfev, n_out=n_out)) if return_info else out


# Bọc an toàn + cache theo (C, w): cùng cửa sổ ở hai chế độ chỉ ước lượng 1 lần; lỗi số ⇒ dự phòng EW-RIE* (được đếm)
_WSP_SAFE = {"fallback": 0, "total": 0}; _WSP_CACHE = {}
def ew_wesper_safe(C, w):
    key = (hashlib.md5(C.tobytes()).hexdigest(), len(w), round(float(w[-1]), 14))
    if key not in _WSP_CACHE:
        _WSP_SAFE["total"] += 1
        try:
            out = ew_wesper_corr(C, w); assert np.isfinite(out).all()
        except Exception:
            _WSP_SAFE["fallback"] += 1; out = ew_rie_corr(C, w)
        _WSP_CACHE[key] = out
    return _WSP_CACHE[key]

_build_estimators_thay = build_estimators           # danh sách của thầy
def build_estimators(cfg):
    E, out = _build_estimators_thay(cfg), {}
    for k, f in E.items():
        if k == "EW-RIE*": out["EW-WeSpeR"] = make_ew(ew_wesper_safe, cfg["HL_EW"])
        out[k] = f
    return out

## 1b. Kiểm thử (chạy trước mọi thay đổi)
1) Trọng số đều ⇒ EW-RIE\* trùng RIE cổ điển. 2) So với oracle trên dữ liệu tổng hợp đã biết C thật.

In [ ]:
rng = np.random.default_rng(0); N_, T_ = 100, 250
X = rng.standard_normal((T_, N_)); Cs = np.corrcoef(X, rowvar=False); lam = np.linalg.eigvalsh(Cs)
z = lam - 0.1j; g = (1 / (z[:, None] - lam[None, :])).mean(1); q = N_ / T_
err = np.max(np.abs(_xi(lam, lam, np.full(T_, 1 / T_), N_, 0.1) - lam / np.abs(1 - q + q*z*g)**2) / lam)
print(f"[Test 1] trọng số đều: sai lệch tương đối max = {err:.1e}"); assert err < 1e-10

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def factor_corr(N, rng):
    sec = rng.integers(0, 8, N); B = np.zeros((N, 9))
    B[:, 0] = rng.uniform(.4, .8, N); B[np.arange(N), 1 + sec] = rng.uniform(.2, .5, N)
    return renorm(B @ B.T + np.diag(rng.uniform(.3, .6, N)))

rows = []
for hl in (30, 60):
    w = ew_weights(T_, hl); acc = {}
    for rep in range(6):
        Ct = factor_corr(N_, rng); X = rng.standard_normal((T_, N_)) @ np.linalg.cholesky(Ct).T
        C, _ = ew_corr(X, w); lam, V = np.linalg.eigh(C)
        Co = renorm((V * np.einsum("ij,jk,ki->i", V.T, Ct, V)) @ V.T)    # oracle RIE
        wo = gmv(Ct); best = wo @ Ct @ wo
        for k, Ch in {"EW-Sample": C, "EW-RIE (q_eff)": rie_corr(C, N_ * (w**2).sum()),
                      "EW-NL-LW": renorm(nl_lw_S(C, 1 / (w**2).sum())), "EW-RIE*": ew_rie_corr(C, w),
                      "Oracle": Co}.items():
            wh = gmv(Ch); acc.setdefault(k, []).append(wh @ Ct @ wh / best)
    rows += [dict(hl=hl, method=k, risk_ratio=np.mean(v)) for k, v in acc.items()]
t2 = pd.DataFrame(rows).pivot(index="method", columns="hl", values="risk_ratio").round(3)
print("[Test 2] rủi ro GMV / rủi ro tối ưu (1 = hoàn hảo), mô hình nhân tố:"); display(t2)
assert (t2.loc["EW-RIE*"] < t2.loc["EW-RIE (q_eff)"]).all()

### ▶ §1b (bổ sung) — Test 2b: kiểm tra bản sửa EW-NL-LW và WeSpeR
Như Test 2 của thầy nhưng thêm half-life 15 (N = 100 > T_eff ≈ 43), bản NL-LW cũ/mới và EW-WeSpeR.

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def factor_corr(N, rng):
    sec = rng.integers(0, 8, N); B = np.zeros((N, 9))
    B[:, 0] = rng.uniform(.4, .8, N); B[np.arange(N), 1 + sec] = rng.uniform(.2, .5, N)
    return renorm(B @ B.T + np.diag(rng.uniform(.3, .6, N)))

rows = []
for hl in (15, 30, 60):
    w = ew_weights(T_, hl); acc = {}
    for rep in range(6):
        Ct = factor_corr(N_, rng); X = rng.standard_normal((T_, N_)) @ np.linalg.cholesky(Ct).T
        C, _ = ew_corr(X, w); lam, V = np.linalg.eigh(C)
        Co = renorm((V * np.einsum("ij,jk,ki->i", V.T, Ct, V)) @ V.T)    # oracle RIE
        wo = gmv(Ct); best = wo @ Ct @ wo
        for k, Ch in {"EW-Sample": C, "EW-RIE (q_eff)": rie_corr(C, N_ * (w**2).sum()),
                      "EW-NL-LW (v2)": renorm(nl_lw_S_v2(C, 1 / (w**2).sum())), "EW-NL-LW": renorm(nl_lw_S(C, 1 / (w**2).sum())),
                      "EW-WeSpeR": ew_wesper_corr(C, w), "EW-RIE*": ew_rie_corr(C, w), "Oracle": Co}.items():
            wh = gmv(Ch); acc.setdefault(k, []).append(wh @ Ct @ wh / best)
    rows += [dict(hl=hl, method=k, risk_ratio=np.mean(v)) for k, v in acc.items()]
t2b = pd.DataFrame(rows).pivot(index="method", columns="hl", values="risk_ratio").round(3)
teff = {hl: 1 / (ew_weights(T_, hl)**2).sum() for hl in (15, 30, 60)}
print("[Test 2b] rủi ro GMV / rủi ro tối ưu (1 = hoàn hảo), mô hình nhân tố, N=100, T=250; T_eff =",
      {h: round(v) for h, v in teff.items()}); display(t2b)
assert (t2b.loc["EW-RIE*", [30, 60]] < t2b.loc["EW-RIE (q_eff)", [30, 60]]).all()
assert (t2b.loc["EW-NL-LW"] <= t2b.loc["EW-NL-LW (v2)"] + 1e-9).all()                 # bản sửa không kém v2
assert np.isclose(t2b.loc["EW-NL-LW", 60], t2b.loc["EW-NL-LW (v2)", 60])              # N < T_eff: trùng v2
assert (t2b.loc["EW-WeSpeR"] / t2b.loc["Oracle"] < 1.03).all()                        # WeSpeR trong 3% của oracle

<!--CMT:test2b-->
(nhận xét điền sau khi xem hình)

## 2. Dữ liệu
Ba lựa chọn qua `DATA_SOURCE`. Kết quả chuẩn: `P` (giá đóng cửa điều chỉnh, dòng = ngày, cột = mã) và `V` (giá trị giao dịch, cùng kích thước).

### ▶ T1.0 — lấy dữ liệu từ GitHub
Bộ `data_hose/` (vnstock 5.2.2, nguồn VCI; lịch sử niêm yết từ HSX + Vietstock + VCI) nằm trong repo public `chinguyenkhac/data_HOSE`. Ô này tải 5 file về `DATA_DIR` nếu chưa có; ô nạp của thầy (ngay dưới) đọc `prices.csv`, `values.csv` từ đó.

In [ ]:
import urllib.request
HOSE_FILES = ["prices.csv", "values.csv", "volumes.csv", "security_master.csv", "missing_tickers.csv"]
os.makedirs(DATA_DIR, exist_ok=True)
for f in HOSE_FILES:
    if not os.path.exists(f"{DATA_DIR}/{f}"):
        urllib.request.urlretrieve(f"{DATA_URL}/{f}", f"{DATA_DIR}/{f}")
print("Nguồn:", DATA_URL)
print("Đã có trong", DATA_DIR, ":", ", ".join(f"{f} ({os.path.getsize(f'{DATA_DIR}/{f}') / 1e6:.1f} MB)" for f in HOSE_FILES))

In [ ]:
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.returncode: print(r.stderr[-500:])
    return r.returncode

def load_github(workdir):
    # Giá ngày 2006–03/2020 (github.com/88d52bdba0366127fffca9dfa93895/vnstock-data);
    # danh sách mã HOSE (github.com/vuthanhdatt/web-indicator, năm 2022 → có survivorship bias)
    px_repo, ls_repo = f"{workdir}/vnstock-data", f"{workdir}/web-indicator"
    if not os.path.isdir(f"{px_repo}/tickers"):
        sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/88d52bdba0366127fffca9dfa93895/vnstock-data {px_repo}")
        sh("git sparse-checkout set tickers", cwd=px_repo)
    if not os.path.isdir(f"{ls_repo}/datas/hose"):
        sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/vuthanhdatt/web-indicator {ls_repo}")
        sh("git sparse-checkout set datas/hose", cwd=ls_repo)
    hose = {f[:-4].lower() for f in os.listdir(f"{ls_repo}/datas/hose")}
    px, val = {}, {}
    for f in sorted(os.listdir(f"{px_repo}/tickers")):
        t = f[:-4]
        if t not in hose: continue
        d = pd.read_csv(f"{px_repo}/tickers/{f}", parse_dates=["date"]).drop_duplicates("date").set_index("date")
        d = d[d["close"] > 0]; px[t.upper()] = d["close"]; val[t.upper()] = d["close"] * d["volume"]
    return pd.DataFrame(px).sort_index(), pd.DataFrame(val)

def load_csv(price_path, value_path=None):
    P = pd.read_csv(price_path, index_col=0, parse_dates=True).sort_index()
    V = pd.read_csv(value_path, index_col=0, parse_dates=True).reindex(P.index) if value_path else P * 0 + 1
    return P, V

def load_vnstock(symbols, start="2006-01-01", end="2025-12-31"):
    # T1: API vnstock thay đổi theo phiên bản – KIỂM TRA lại trước khi dùng (pip install -U vnstock).
    from vnstock import Vnstock
    px, val = {}, {}
    for s in symbols:
        try:
            d = Vnstock().stock(symbol=s, source="VCI").quote.history(start=start, end=end, interval="1D")
            d = d.set_index(pd.to_datetime(d["time"]))
            px[s] = d["close"]; val[s] = d["close"] * d["volume"]   # LƯU Ý: kiểm tra giá đã điều chỉnh chưa
        except Exception as e:
            print(s, "lỗi:", e)
        time.sleep(0.3)
    return pd.DataFrame(px).sort_index(), pd.DataFrame(val).sort_index()

if DATA_SOURCE == "github":
    P, V = load_github(WORKDIR)
elif DATA_SOURCE == "csv":
    P, V = load_csv(f"{DATA_DIR}/prices.csv", f"{DATA_DIR}/values.csv")   # ▶ T1: dữ liệu tải từ GitHub
else:
    P, V = load_vnstock(["VNM", "FPT", "VCB"])         # T1: thay bằng danh sách đầy đủ, gồm mã đã hủy niêm yết

P = P[P.notna().sum(1) >= 20]; V = V.reindex(P.index)
print(f"{P.shape[1]} mã, {P.shape[0]} phiên: {P.index[0].date()} → {P.index[-1].date()}")

### ▶ T1.1 — chỉ giữ giá trong khoảng niêm yết HOSE
`security_master.csv`: mỗi dòng = một giai đoạn niêm yết (ngày niêm yết/hủy, lý do). Giá nằm ngoài khoảng này bị loại; **mã hủy niêm yết được giữ** trong thời gian còn niêm yết, nên danh sách mã không bị thiên về các công ty còn sống.

In [ ]:
# Đọc thêm khối lượng, lịch sử niêm yết (security_master) và danh sách mã thiếu giá
P_RAW = P.copy()                                                         # P, V do ô nạp của thầy ở trên
Q = pd.read_csv(f"{DATA_DIR}/volumes.csv", index_col=0, parse_dates=True).reindex(index=P_RAW.index, columns=P_RAW.columns)
MASTER_RAW = pd.read_csv(f"{DATA_DIR}/security_master.csv", parse_dates=["listing_date", "delisting_date"])
MISSING = pd.read_csv(f"{DATA_DIR}/missing_tickers.csv")
MASTER = MASTER_RAW.drop_duplicates(subset=["ticker", "listing_date", "delisting_date"]).reset_index(drop=True)

# Mặt nạ "đang niêm yết": True nếu ngày nằm trong [ngày niêm yết, ngày hủy] của một giai đoạn niêm yết (một mã có thể có nhiều giai đoạn)
INWIN = pd.DataFrame(False, index=P_RAW.index, columns=P_RAW.columns)
for r in MASTER.itertuples():
    if r.ticker in INWIN:
        end_ok = (INWIN.index <= r.delisting_date) if pd.notna(r.delisting_date) else True
        INWIN.loc[(INWIN.index >= r.listing_date) & end_ok, r.ticker] = True

OUTSIDE = P_RAW.notna() & ~INWIN                                         # quan sát giá nằm ngoài khoảng niêm yết
P = P_RAW.where(INWIN)
V = V.reindex(index=P_RAW.index, columns=P_RAW.columns).where(INWIN)
Q = Q.where(INWIN)
P = P[P.notna().sum(1) >= 20]; V = V.reindex(P.index); Q = Q.reindex(P.index)

# Mã đã rời HOSE (không còn niêm yết ở giai đoạn nào)
DELISTED = set(MASTER.loc[MASTER.delisting_date.notna(), "ticker"]) - set(MASTER.loc[MASTER.delisting_date.isna(), "ticker"])
print(f"{P.shape[1]} mã có giá, {P.shape[0]} phiên: {P.index[0].date()} → {P.index[-1].date()} | "
      f"thiếu giá: {len(MISSING)} mã | quan sát ngoài khoảng niêm yết (đã loại): {int(OUTSIDE.sum().sum())}")

In [ ]:
# Kiểm tra chất lượng dữ liệu
R_all = np.log(P).diff()
print("Tỷ lệ |r| > ngưỡng:", f"{(R_all.abs() > CONFIG['MAX_ABS_RET']).sum().sum() / R_all.notna().sum().sum():.4%}")
print("Tỷ lệ phiên giá không đổi (r = 0):", f"{(R_all == 0).sum().sum() / R_all.notna().sum().sum():.1%}")
P.notna().groupby(P.index.year).any().sum(axis=1).plot.bar(figsize=(9, 2.5), title="Số mã có dữ liệu theo năm"); plt.show()

### ▶ T1 – Kiểm tra chất lượng dữ liệu và lịch sử niêm yết
**Câu hỏi:** dữ liệu có đủ sạch để backtest, và còn thiên lệch do bỏ sót mã hủy niêm yết hay do mã đứng giá không?
- **T1a** tổng quan và các phép kiểm tra cơ bản · **T1b** số cổ phiếu niêm yết / hủy niêm yết theo năm · **T1c** mã đứng giá và thanh khoản · **T1d** số mã đủ điều kiện mỗi kỳ theo ngưỡng lọc.

In [ ]:
# T1a. Tổng quan dữ liệu sau khi lọc theo khoảng niêm yết
R_q = np.log(P).diff(); n_ret = int(R_q.notna().sum().sum()); in_win = INWIN.reindex(P.index)
dl = MASTER.dropna(subset=["delisting_date"]); has_px = dl.ticker.isin(P.columns)

TQ = pd.Series({
    "Khoảng thời gian": f"{P.index[0].date()} → {P.index[-1].date()} ({len(P)} phiên)",
    "Số mã trong security_master / có giá / thiếu giá": f"{MASTER.ticker.nunique()} / {P.shape[1]} / {len(MISSING)}",
    "Lượt hủy niêm yết (có giá / thiếu giá)": f"{len(dl)} ({int(has_px.sum())} / {int((~has_px).sum())})",
    "Quan sát ngoài khoảng niêm yết (đã loại)": f"{int(OUTSIDE.sum().sum())} (mã: {', '.join(OUTSIDE.sum()[OUTSIDE.sum() > 0].index)})",
    "% ô có giá trong khoảng niêm yết": f"{100 * (P.notna() & in_win).sum().sum() / in_win.sum().sum():.1f}%",
    "% phiên đứng giá (r = 0)": f"{100 * (R_q == 0).sum().sum() / n_ret:.1f}%",
    "Phiên |r| > 15% (đặt = 0) / khối lượng = 0 mà có giá / giá ≤ 0":
        f"{int((R_q.abs() > CONFIG['MAX_ABS_RET']).sum().sum())} / {int(((Q == 0) & P.notna()).sum().sum())} / {int((P <= 0).sum().sum())}"})
display(TQ.to_frame("giá trị"))

<!--CMT:t1a-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T1b. Số cổ phiếu niêm yết (cuối năm) và số cổ phiếu hủy niêm yết theo năm, tách theo lý do
REASON = {"bat_buoc": "bắt buộc", "hop_nhat": "hợp nhất/sáp nhập", "tu_nguyen": "tự nguyện", "chuyen_san": "chuyển sàn", "chua_ro": "chưa rõ"}
REASON_COL = {"bắt buộc": "#eb6834", "hợp nhất/sáp nhập": "#2a78d6", "tự nguyện": "#1baf7a", "chuyển sàn": "#8e44ad", "chưa rõ": "#9aa0a6"}
years = list(range(P.index[0].year, P.index[-1].year + 1))
year_end = {y: min(pd.Timestamp(f"{y}-12-31"), P.index[-1]) for y in years}

def n_listed(day):
    """Số mã đang niêm yết vào ngày `day` theo security_master."""
    return sum(1 for r in MASTER.itertuples() if r.listing_date <= day and (pd.isna(r.delisting_date) or r.delisting_date >= day))

listed = pd.Series({y: n_listed(year_end[y]) for y in years})
by_reason = (dl.assign(year=dl.delisting_date.dt.year, reason=dl.reason.map(REASON).fillna("chưa rõ"))
               .pivot_table(index="year", columns="reason", values="ticker", aggfunc="count", fill_value=0)
               .reindex(years, fill_value=0))
by_reason = by_reason[by_reason.sum().sort_values(ascending=False).index]          # lý do nhiều nhất ở dưới cùng

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
bars = ax[0].bar(years, listed.values, color="#2a78d6", width=.7)
ax[0].bar_label(bars, fontsize=8, padding=2)
ax[0].set(title="Tổng số cổ phiếu niêm yết trên HOSE (cuối năm)", xticks=years, ylim=(0, listed.max() * 1.12), ylabel="số mã")
ax[0].set_xlabel("2026: tính đến 11/2026")

bottom = np.zeros(len(years))
for reason in by_reason.columns:
    ax[1].bar(years, by_reason[reason], bottom=bottom, width=.7, color=REASON_COL[reason], label=reason); bottom += by_reason[reason].values
for x, v in zip(years, bottom): ax[1].text(x, v + .3, str(int(v)), ha="center", fontsize=8)
ax[1].set(title="Số cổ phiếu hủy niêm yết (rời HOSE) theo năm và lý do", xticks=years, ylim=(0, bottom.max() * 1.15), ylabel="số mã")
ax[1].legend(ncol=5, loc="upper center", bbox_to_anchor=(.5, -.1), fontsize=8)
fig.tight_layout(); plt.show()
print("Lý do hủy:", dl.reason.map(REASON).fillna("chưa rõ").value_counts().to_dict())

<!--CMT:t1b-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T1c. Mã đứng giá: tỷ lệ phiên r = 0 của từng mã và liên hệ với thanh khoản
n_obs = R_q.notna().sum()
zero_share = ((R_q == 0).sum() / n_obs)[n_obs >= 250] * 100                        # % phiên r = 0 của mỗi mã (mã có ≥ 250 phiên)
liq = V.median()[zero_share.index] / 1e9                                           # GTGD trung vị (tỷ đồng/phiên)
quint = pd.qcut(liq.rank(method="first"), 5, labels=["Q1\n(kém nhất)", "Q2", "Q3", "Q4", "Q5\n(tốt nhất)"])
zmax = 100 * CONFIG["ZERO_MAX"]

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
ax[0].hist(zero_share, bins=35, color=GREY, edgecolor="white")
ax[0].axvline(zmax, color="#eb6834", lw=2); ax[0].axvspan(zmax, zero_share.max() + 1, color="#eb6834", alpha=.08)
ax[0].text(zmax + .8, ax[0].get_ylim()[1] * .9, f"{int((zero_share > zmax).sum())}/{len(zero_share)} mã\nvượt ZERO_MAX = {zmax:.0f}%", color="#eb6834", fontsize=9, va="top")
ax[0].set(title="Phân bố % phiên đứng giá theo mã", xlabel="% phiên r = 0 của mã", ylabel="số mã")

med = zero_share.groupby(quint, observed=True).median()
bars = ax[1].bar(range(5), med.values, color=["#eb6834", "#f0925c", "#c9b27a", "#7fbf9c", "#1baf7a"])
ax[1].bar_label(bars, fmt="%.0f%%", fontsize=9, padding=2); ax[1].axhline(zmax, color="#eb6834", lw=1.2, ls="--")
ax[1].set(title="Thanh khoản kém ⇒ đứng giá nhiều hơn", xticks=range(5), xticklabels=med.index, ylabel="trung vị % phiên r = 0", ylim=(0, med.max() * 1.2))

ax[2].scatter(liq, zero_share, s=10, alpha=.55, color="#52514e")
ax[2].axhline(zmax, color="#eb6834", lw=1.2, ls="--"); ax[2].set_xscale("log")
ax[2].set(title="Thanh khoản và đứng giá", xlabel="GTGD trung vị (tỷ đồng/phiên, thang log)", ylabel="% phiên r = 0")
fig.tight_layout(); plt.show()
print(f"Tương quan hạng (GTGD, % r = 0) = {liq.rank().corr(zero_share.rank()):.2f} | trung vị theo nhóm thanh khoản:",
      {str(k).replace(chr(10), ' '): round(v, 1) for k, v in med.items()})

<!--CMT:t1c-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T1d. Số mã đủ điều kiện vào vũ trụ mỗi kỳ tái cân bằng, theo ngưỡng lọc đứng giá (cơ sở chọn ZERO_MAX)
Rm = R_q.mask(R_q.abs() > CONFIG["MAX_ABS_RET"], 0.0); Wn = CONFIG["WINDOW"]; rows, dates_ = [], []
for s in range(Wn + 1, len(P) - 1, CONFIG["REB"]):
    win = Rm.iloc[s - Wn:s]
    ok = (win.notna().mean() >= .95) & P.iloc[s - 1].notna()                       # đủ dữ liệu và có giá hôm trước (điều kiện của thầy)
    zs = (win == 0).mean()                                                         # tỷ lệ r = 0 trong cửa sổ quá khứ
    rows.append({z: int((ok & (zs <= z)).sum()) for z in (1.0, 0.2, 0.1)}); dates_.append(P.index[s])
EL = pd.DataFrame(rows, index=dates_)
TE = pd.DataFrame({"ZERO_MAX": ["không lọc", "20% (chuẩn)", "10%"],
                   f"số kỳ đủ N = {CONFIG['N']} mã (/{len(EL)})": [(EL[z] >= CONFIG["N"]).sum() for z in EL],
                   "số mã đủ điều kiện: trung vị": [int(EL[z].median()) for z in EL], "nhỏ nhất": [int(EL[z].min()) for z in EL]}).set_index("ZERO_MAX")
display(TE)

fig, ax = plt.subplots(figsize=(11.5, 3.8))
for z, lab, c, lw in ((1.0, "không lọc", GREY, 2), (0.2, "lọc 20% (chuẩn)", "#1baf7a", 2.6), (0.1, "lọc 10%", "#eda100", 2)):
    ax.plot(EL.index, EL[z], color=c, lw=lw, label=f"{lab}: {(EL[z] >= CONFIG['N']).sum()}/{len(EL)} kỳ đủ N")
ax.axhline(CONFIG["N"], color="#eb6834", ls="--", lw=1.3); ax.text(EL.index[-1], CONFIG["N"] + 6, f"N = {CONFIG['N']}", color="#eb6834", ha="right")
ax.set(title="Số mã đủ điều kiện mỗi kỳ tái cân bằng", ylabel="số mã"); ax.legend(loc="upper left")
fig.tight_layout(); plt.show()

<!--CMT:t1d-->
(nhận xét điền sau khi xem hình)

## 3. Danh mục & backtest
Vũ trụ động: mỗi kỳ chọn N mã có giá trị GD trung vị cao nhất trong cửa sổ (≥95% phiên có dữ liệu). Mã ngừng GD ngoài mẫu → lợi suất 0.

In [ ]:
def gmv_unconstrained(S):
    w = np.linalg.solve(S, np.ones(len(S))); return w / w.sum()

def gmv_long_only(S, iters=3000):
    # FISTA + chiếu lên đơn hình (khớp SLSQP, nhanh hơn)
    n = len(S)
    def proj(v):
        u = np.sort(v)[::-1]; css = np.cumsum(u) - 1
        k = np.nonzero(u - css / np.arange(1, n + 1) > 0)[0][-1]; return np.maximum(v - css[k] / (k + 1), 0)
    L = 2 * np.linalg.eigvalsh(S)[-1]; w = y = np.full(n, 1 / n); t = 1.0
    for _ in range(iters):
        wn = proj(y - 2 * S @ y / L); tn = (1 + np.sqrt(1 + 4*t*t)) / 2
        y = wn + (t - 1) / tn * (wn - w)
        if np.abs(wn - w).max() < 1e-9: return wn
        w, t = wn, tn
    return w

PERIODS = [("2008-09 Khủng hoảng TC", "2008-01-01", "2009-12-31"), ("2010-12 Lạm phát/nợ xấu", "2010-01-01", "2012-12-31"),
           ("2013-17 Ổn định", "2013-01-01", "2017-12-31"), ("2018-19 Điều chỉnh", "2018-01-01", "2019-12-31"),
           ("2020Q1 COVID", "2020-01-01", "2020-03-31"), ("2020-21 Bùng nổ", "2020-04-01", "2021-12-31"),
           ("2022 Sụt giảm", "2022-01-01", "2022-12-31"), ("2023-25", "2023-01-01", "2025-12-31")]

def run_backtest(P, V, estimators, cfg, long_only):
    N, W, REB = cfg["N"], cfg["WINDOW"], cfg["REB"]
    R = np.log(P).diff(); R = np.expm1(R.mask(R.abs() > cfg["MAX_ABS_RET"], 0.0))
    dates = R.index; names = list(estimators)
    rets = {k: [] for k in names}; idx = []; reb_dates = []
    Wp = {k: pd.Series(dtype=float) for k in names}
    st = {k: dict(turn=[], neff=[], wmax=[], short=[]) for k in names}
    for s in range(W + 1, len(dates) - 1, REB):
        win = R.iloc[s - W:s]; ok = (win.notna().mean() >= .95) & P.iloc[s - 1].notna()
        if ok.sum() < N: continue
        tick = V.iloc[s - W:s].loc[:, ok].median().sort_values(ascending=False).index[:N]
        X = win[tick].fillna(0).values; Y = R.iloc[s:s + REB][tick].fillna(0).values
        idx.extend(dates[s:s + REB]); reb_dates.append(dates[s])
        for k in names:
            S = estimators[k](X)
            w = gmv_long_only(S) if long_only else gmv_unconstrained(S)
            ws = pd.Series(w, index=tick)
            if len(Wp[k]):
                u = ws.index.union(Wp[k].index)
                st[k]["turn"].append((ws.reindex(u).fillna(0) - Wp[k].reindex(u).fillna(0)).abs().sum())
            Wp[k] = ws; rets[k].extend(Y @ w)
            st[k]["neff"].append(1 / (w**2).sum()); st[k]["wmax"].append(w.max()); st[k]["short"].append(np.clip(-w, 0, None).sum())
    out = {}
    for k in names:
        r = pd.Series(rets[k], index=idx); rn = r.copy()
        rn.loc[reb_dates[1:]] -= cfg["COST"] * np.array(st[k]["turn"])
        out[k] = dict(r=r, r_net=rn, m={
            "Vol %": r.std() * np.sqrt(252) * 100, "LN/năm %": r.mean() * 252 * 100,
            "Sharpe": r.mean() / r.std() * np.sqrt(252), "Sharpe sau phí": rn.mean() / rn.std() * np.sqrt(252),
            "Turnover/kỳ": np.mean(st[k]["turn"]), "Số mã hiệu dụng": np.mean(st[k]["neff"]),
            "Tỷ trọng max %": np.mean(st[k]["wmax"]) * 100, "Short %": np.mean(st[k]["short"]) * 100})
    return out

def block_bootstrap_ci(ra, rb, block=21, B=2000, seed=0):
    # CI 95% cho Vol(a) − Vol(b), %/năm. (T3: bổ sung kiểm định Ledoit–Wolf 2011)
    rng = np.random.default_rng(seed); n = len(ra); nb = n // block; d = np.empty(B)
    for i in range(B):
        ix = (rng.integers(0, n - block, nb)[:, None] + np.arange(block)).ravel()
        d[i] = (ra[ix].std() - rb[ix].std()) * np.sqrt(252) * 100
    return np.percentile(d, [2.5, 97.5])

def summarize(out, pairs):
    tab = pd.DataFrame({k: o["m"] for k, o in out.items()}).T
    per = pd.DataFrame({k: {p: o["r"][s:e].std() * np.sqrt(252) * 100 for p, s, e in PERIODS if len(o["r"][s:e]) > 20}
                        for k, o in out.items()}).T
    ci = pd.DataFrame({f"{a} vs {b}": block_bootstrap_ci(out[a]["r"].values, out[b]["r"].values)
                       for a, b in pairs if a in out and b in out}, index=["CI 2.5%", "CI 97.5%"]).T
    ci["Có ý nghĩa"] = (ci["CI 97.5%"] < 0) | (ci["CI 2.5%"] > 0)
    return tab, per, ci

### ▶ §3b. Lọc mã đứng giá (`ZERO_MAX`) — không sửa `run_backtest` của thầy
`run_backtest` bên dưới được **sinh từ chính mã nguồn của thầy** (`inspect.getsource`), chỉ chèn một dòng ngay trước `if ok.sum() < N: continue`:
`ok &= (win == 0).mean() <= cfg.get("ZERO_MAX", 1.0)` — loại mã có quá nhiều phiên r = 0 trong **cửa sổ quá khứ** (không dùng tương lai); kỳ không đủ N mã bị bỏ, không hạ N. Với `ZERO_MAX = 1.0` kết quả trùng hệt bản của thầy (assert bên dưới). `PERIODS` cập nhật theo mẫu 2016–2026.

In [ ]:
import inspect
ULOG = []                                                # nhật ký vũ trụ (danh sách mã) mỗi kỳ
_src = inspect.getsource(run_backtest)
_chk = "        if ok.sum() < N: continue\n"
_sel = "        tick = V.iloc[s - W:s].loc[:, ok].median().sort_values(ascending=False).index[:N]\n"
assert _src.count(_chk) == 1 and _src.count(_sel) == 1, "mã nguồn run_backtest của thầy đã đổi"
exec(_src.replace("def run_backtest(", "def run_backtest_x(")
         .replace(_chk, '        ok &= (win == 0).mean() <= cfg.get("ZERO_MAX", 1.0)          # ▶ lọc mã đứng giá (cửa sổ quá khứ)\n' + _chk)
         .replace(_sel, _sel + "        ULOG.append(dict(date=dates[s], n_eligible=int(ok.sum()), tickers=list(tick)))\n"))
run_backtest_thay = run_backtest                         # bản gốc của thầy (để đối chiếu)
def run_backtest(*a, **k):                               # các ô của thầy gọi run_backtest → dùng bản có lọc
    ULOG.clear(); return run_backtest_x(*a, **k)

PERIODS = [("2016-17 Ổn định", "2016-01-01", "2017-12-31"), ("2018-19 Điều chỉnh", "2018-01-01", "2019-12-31"),
           ("2020Q1 COVID", "2020-01-01", "2020-03-31"), ("2020-21 Bùng nổ", "2020-04-01", "2021-12-31"),
           ("2022 Sụt giảm", "2022-01-01", "2022-12-31"), ("2023-24 Phục hồi", "2023-01-01", "2024-12-31"),
           ("2025-26", "2025-01-01", "2026-12-31")]

_p, _v = P.iloc[-900:], V.iloc[-900:]                    # kiểm tra: ZERO_MAX = 1.0 ⇒ trùng hệt bản của thầy
_a = run_backtest_thay(_p, _v, {"NL-LW": est_nllw}, CONFIG, True)["NL-LW"]["r"]
_b = run_backtest(_p, _v, {"NL-LW": est_nllw}, dict(CONFIG, ZERO_MAX=1.0), True)["NL-LW"]["r"]
assert _a.index.equals(_b.index) and np.array_equal(_a.values, _b.values)
print(f"run_backtest (có lọc) ≡ run_backtest của thầy khi ZERO_MAX = 1.0 ✔ | cấu hình chuẩn: ZERO_MAX = {CONFIG['ZERO_MAX']:.0%}")

## 4. Chạy chính
Thời gian: chế độ đầy đủ trên Colab khoảng 15–30 phút (10 bộ ước lượng × 2 chế độ). Đặt `QUICK=True` để chạy thử nhanh.

In [ ]:
ESTIMATORS = build_estimators(CONFIG)
PAIRS = [("EW-RIE*", "EW-RIE (q_eff)"), ("EW-RIE*", "EW-NL-LW"), ("EW-RIE*", "NL-LW"), ("EW-RIE*", "RIE"),
         ("NL-LW", "Ledoit-Wolf"), ("VS-NL-LW", "NL-LW")]
RESULTS = {}
for long_only in (False, True):
    mode = "long_only" if long_only else "short_ok"; t0 = time.time()
    out = run_backtest(P, V, ESTIMATORS, CONFIG, long_only)
    RESULTS[mode] = dict(out=out, **dict(zip(["table", "periods", "ci"], summarize(out, PAIRS))))
    print(f"\n===== {mode} | {time.time() - t0:.0f}s =====")
    display(RESULTS[mode]["table"].round(2))
    display(RESULTS[mode]["periods"].round(2))
    display(RESULTS[mode]["ci"].round(3))
print("Newton thất bại:", _STATS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, mode, ttl in [(ax[0], "short_ok", "Cho phép bán khống"), (ax[1], "long_only", "Chỉ mua")]:
    v = RESULTS[mode]["table"]["Vol %"].sort_values()
    a.barh(v.index, v.values, color=["#1baf7a" if k == "EW-RIE*" else "#9aa0a6" for k in v.index])
    a.set_xlim(v.min() - 1, v.max() + 0.5); a.set_title(f"{ttl}: rủi ro ngoài mẫu (%/năm, trục không từ 0)")
    for i, x in enumerate(v.values): a.text(x, i, f" {x:.2f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

<!--CMT:main_bar-->
(nhận xét điền sau khi xem hình)

### ▶ §4b. Vol theo giai đoạn thị trường
Cùng kết quả §4, nhìn theo từng giai đoạn để thấy phương pháp nào hơn ở đâu.

In [ ]:
# ▶ Vol theo giai đoạn thị trường: màu = chênh so với NL-LW (xanh = thấp hơn NL-LW), số = Vol %/năm
SHOW = ["NL-LW", "VS-NL-LW", "EW-NL-LW", "EW-WeSpeR", "EW-RIE*"]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.9))
for ax, (mode, ttl) in zip(axes, (("short_ok", "Bán khống"), ("long_only", "Chỉ mua"))):
    per = RESULTS[mode]["periods"].loc[SHOW]                                       # hàng = phương pháp, cột = giai đoạn
    rel = (per / per.loc["NL-LW"] - 1) * 100                                       # % chênh so với NL-LW cùng giai đoạn
    lim = np.abs(rel.values).max()
    ax.imshow(rel.values, cmap="RdYlGn_r", norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
    ax.set_xticks(range(per.shape[1]), [c.replace(" ", "\n", 1) for c in per.columns], fontsize=8)
    ax.set_yticks(range(len(SHOW)), SHOW); ax.grid(False)
    for (i, j), v in np.ndenumerate(per.values):
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=9, fontweight="bold" if SHOW[i] == "EW-RIE*" else "normal")
    ax.set_title(f"{ttl}: Vol %/năm (màu: % chênh so với NL-LW)")
fig.tight_layout(); plt.show()

<!--CMT:main_periods-->
(nhận xét điền sau khi xem hình)

In [ ]:
# Lưu kết quả (tải về hoặc chép vào Drive)
with pd.ExcelWriter(f"{WORKDIR}/results_N{CONFIG['N']}_T{CONFIG['WINDOW']}_h{CONFIG['HL_EW']}.xlsx") as xw:
    for mode, d in RESULTS.items():
        d["table"].round(4).to_excel(xw, sheet_name=f"{mode}_tong_hop")
        d["periods"].round(3).to_excel(xw, sheet_name=f"{mode}_giai_doan")
        d["ci"].round(4).to_excel(xw, sheet_name=f"{mode}_CI")
    pd.Series(CONFIG).to_frame("giá trị").to_excel(xw, sheet_name="CONFIG")
print("Đã lưu vào", WORKDIR)

## 5. Độ nhạy half-life (T6 mở rộng thêm N, T)

In [ ]:
RUN_SENS = True    # đặt True để chạy (~10–20 phút)
if RUN_SENS:
    rows = []
    for hl in (30, 60, 120):
        est = {"EW-NL-LW": make_ew(lambda C, w: renorm(nl_lw_S(C, 1 / (w**2).sum())), hl), "EW-RIE*": make_ew(ew_rie_corr, hl)}
        for lo in (False, True):
            o = run_backtest(P, V, est, CONFIG, lo)
            l, u = block_bootstrap_ci(o["EW-RIE*"]["r"].values, o["EW-NL-LW"]["r"].values)
            rows.append(dict(hl=hl, mode="long_only" if lo else "short_ok", vol_EWNLLW=o["EW-NL-LW"]["m"]["Vol %"],
                             vol_EWRIE=o["EW-RIE*"]["m"]["Vol %"], CI_lo=l, CI_hi=u))
    display(pd.DataFrame(rows).round(3))

<!--CMT:sens-->
(nhận xét điền sau khi xem hình)

## 6. Việc của nhóm (khung sẵn – điền tiếp)

### ▶ §6.0 Tiện ích chung cho T2, T3, T5, T6
Chạy backtest có cache (`run`), chỉ số, định dạng bảng; tái dùng `run_backtest`, `summarize`, `block_bootstrap_ci` của thầy.

In [ ]:
import hashlib, json, platform, importlib.util
from IPython.display import Markdown, display
from scipy.stats import norm
if importlib.util.find_spec("arch") is None:        # Colab chưa có sẵn `arch` (dùng cho DCC-GARCH ở T3)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arch"], check=True)

SEED = 0; np.random.seed(SEED)
OUT_DIR = "./results_v5"; CACHE = f"{OUT_DIR}/cache"; os.makedirs(CACHE, exist_ok=True)
RECOMPUTE = False                    # True: chạy lại mọi backtest; False: dùng cache nếu cấu hình giống hệt
CORE = ["NL-LW", "EW-NL-LW", "EW-RIE*"]                                    # 3 phương pháp chính của T2, T6
MODES = {"long_only": True, "short_ok": False}                             # tên chế độ → cờ long_only
VN = {"long_only": "Chỉ mua", "short_ok": "Bán khống"}
EST_CORE = {k: ESTIMATORS[k] for k in CORE}


# ---------- kiểm tra Σ và đo thời gian ----------
TIMES = {}
def check_cov(S, name=""):
    """Σ phải hữu hạn, đối xứng và bán xác định dương."""
    ev = np.linalg.eigvalsh((S + S.T) / 2)
    assert np.isfinite(S).all() and np.allclose(S, S.T, rtol=0, atol=1e-10 * np.abs(S).max()) and ev[0] > -1e-10 * ev[-1], name
    return ev

def timed(name, f):
    def wrapped(X):
        t0 = time.perf_counter(); S = f(X)
        TIMES.setdefault(name, []).append(time.perf_counter() - t0); check_cov(S, name); return S
    return wrapped


# ---------- chạy backtest có cache ----------
def run(est, cfg=None, long_only=True, data=None):
    """Chạy `run_backtest` của thầy; lưu cache theo (tên bộ ước lượng, cấu hình, chế độ).
    Trả về dict: out (kết quả từng phương pháp), ulog (vũ trụ mỗi kỳ), sec (giây/lần ước lượng), newton (số lần Newton thất bại, tổng)."""
    cfg = cfg or CONFIG; Pd, Vd = data if data is not None else (P, V)
    key = hashlib.md5(repr((sorted(est), sorted(cfg.items()), long_only)).encode()).hexdigest()[:12]
    path = f"{CACHE}/{key}.pkl"
    if os.path.exists(path) and not RECOMPUTE:
        d = pd.read_pickle(path)
    else:
        TIMES.clear(); ULOG.clear(); fail0, tot0 = _STATS["fail"], _STATS["total"]
        out = run_backtest(Pd, Vd, {k: timed(k, f) for k, f in est.items()}, cfg, long_only)
        d = dict(out=out, ulog=list(ULOG), sec={k: np.mean(v) for k, v in TIMES.items()},
                 newton=(_STATS["fail"] - fail0, _STATS["total"] - tot0))
        pd.to_pickle(d, path)
    idx = [o["r"].index for o in d["out"].values()]
    assert all(i.equals(idx[0]) for i in idx), "các phương pháp phải cùng ngày ngoài mẫu"
    return d


# ---------- chỉ số ----------
ann = lambda r: (r.std() * np.sqrt(252) * 100, r.mean() / r.std() * np.sqrt(252))        # (Vol %/năm, Sharpe)
net = lambda o, c: o["r"] - (c / CONFIG["COST"]) * (o["r"] - o["r_net"])                  # lợi suất sau phí ở mức phí c

def metrics(o, cost=None, dates=None):
    """Chỉ số của một phương pháp; `cost`: phí một chiều; `dates`: chỉ tính trên các ngày này."""
    r, rn = o["r"], net(o, cost or CONFIG["COST"])
    if dates is not None: r, rn = r.loc[dates], rn.loc[dates]
    return {"Vol %": ann(r)[0], "Sharpe": ann(r)[1], "Sharpe sau phí": ann(rn)[1], "LN/năm %": r.mean() * 252 * 100,
            "Turnover/kỳ": o["m"]["Turnover/kỳ"], "Số mã hiệu dụng": o["m"]["Số mã hiệu dụng"], "Số ngày OOS": len(r)}


# ---------- định dạng ----------
f2 = lambda x, d=2: f"{x:.{d}f}".replace(".", ",")
pv = lambda p: "< 0,001" if p < 1e-3 else f2(p, 3)
def show(df, d=3):
    """In bảng với dấu phẩy thập phân."""
    display(df.style.format(precision=d, decimal=",", thousands=" ", na_rep="–"))
def save(df, name):
    assert np.isfinite(df.select_dtypes("number").values).all(), f"{name}: có NaN/inf"; df.to_csv(f"{OUT_DIR}/{name}.csv")


# ---------- dữ liệu dùng chung cho T2–T6 ----------
R_BT = np.log(P).diff(); R_BT = np.expm1(R_BT.mask(R_BT.abs() > CONFIG["MAX_ABS_RET"], 0.0))       # lợi suất đúng như trong run_backtest
POS = {d: i for i, d in enumerate(R_BT.index)}
BASE = run(EST_CORE, CONFIG, True)                                                                # cấu hình chuẩn, chỉ mua
assert all(np.allclose(BASE["out"][k]["r"].values, RESULTS["long_only"]["out"][k]["r"].values) for k in CORE), "run() khác §4"
U0 = BASE["ulog"]

json.dump(dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, arch=__import__("arch").__version__, SEED=SEED, **CONFIG),
          open(f"{OUT_DIR}/ENV_CONFIG.json", "w"), indent=1)
print(f"Dữ liệu: {P.shape[1]} mã, {P.shape[0]} phiên | ngoài mẫu: {len(U0)} kỳ, {len(BASE['out']['EW-RIE*']['r'])} phiên "
      f"({U0[0]['date'].date()} → {BASE['out']['EW-RIE*']['r'].index[-1].date()})")
print(f"Mã đủ điều kiện mỗi kỳ: {min(u['n_eligible'] for u in U0)}–{max(u['n_eligible'] for u in U0)} (N = {CONFIG['N']}, ZERO_MAX = {CONFIG['ZERO_MAX']:.0%}) | run() trùng §4: OK")

### T2 – Độ vững: chi phí giao dịch, tần suất tái cân bằng, phiên không giao dịch
**Câu hỏi:** hiệu quả của EW-RIE* có còn khi thay đổi điều kiện vận hành danh mục?
- **T2a** lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%} (9 kịch bản × 2 chế độ).
- **T2b** phiên lợi suất 0: không lọc vs lọc mã có > 20% / > 10% phiên r = 0 trong cửa sổ quá khứ.
- **T2c** quy ước phí: COST một chiều vs phí thực tế mua 0,15% / bán 0,25% (thuế bán 0,1%); unit test turnover × cost.

In [ ]:
# ---- T2: Độ vững ----
# Gợi ý: lặp CONFIG["REB"] in (5, 21, 63); CONFIG["COST"] in (0.0015, 0.0025, 0.004);
# xử lý phiên không giao dịch: thay r=0 bằng NaN rồi dùng hiệp phương sai theo cặp, hoặc lọc mã có >X% phiên r=0.
# for reb in (5, 21, 63):
#     cfg = dict(CONFIG, REB=reb); out = run_backtest(P, V, {"NL-LW": est_nllw, "EW-RIE*": ESTIMATORS["EW-RIE*"]}, cfg, False)

In [ ]:
# T2a. Lưới REB ∈ {5, 21, 63} × COST ∈ {0,15%; 0,25%; 0,40%}, cả hai chế độ.
# COST không làm đổi tỷ trọng nên mỗi (REB, chế độ) chỉ chạy 1 backtest; Sharpe sau phí ở từng COST tính lại từ chuỗi turnover.
REBS, COSTS = (5, 21, 63), (0.0015, 0.0025, 0.004)
T2 = {(reb, mode): run(EST_CORE, dict(CONFIG, REB=reb), lo) for reb in REBS for mode, lo in MODES.items()}

rows = []
for (reb, mode), d in T2.items():
    for k, o in d["out"].items():
        for c in COSTS:
            rows.append({"mode": mode, "REB": reb, "COST": c, "method": k, **metrics(o, c), "Turnover/năm": o["m"]["Turnover/kỳ"] * 252 / reb})
T2A = pd.DataFrame(rows); save(T2A, "T2a_reb_cost_grid")

# kiểm tra hợp lý: phí tăng thì Sharpe sau phí không được tăng
mono = T2A.sort_values("COST").groupby(["method", "mode", "REB"])["Sharpe sau phí"].apply(lambda s: (s.diff().dropna() <= 1e-12).all())
assert mono.all(), "COST tăng mà Sharpe sau phí tăng?"

# bảng (phí gốc COST = 0,25%)
at_base = T2A[T2A.COST == CONFIG["COST"]]
tab = (at_base.pivot_table(index=["mode", "REB"], columns="method", values=["Vol %", "Sharpe sau phí", "Turnover/năm"])
              [["Vol %", "Sharpe sau phí", "Turnover/năm"]].reindex(columns=CORE, level=1))
print(f"Bảng T2a – COST = {CONFIG['COST']:.2%} (Vol %/năm gộp; Sharpe sau phí; turnover năm hóa = Σ|Δw|/kỳ × 252/REB)")
show(tab.rename(index=VN, level=0))

# hình: hàng = chế độ; cột 1 = Vol theo REB; cột 2–4 = Sharpe sau phí theo REB × COST (cùng thang màu trong một hàng)
fig, axes = plt.subplots(2, 4, figsize=(15.5, 7.4), layout="constrained", gridspec_kw=dict(width_ratios=[1.3, 1, 1, 1]))
for i, mode in enumerate(MODES):
    ax = axes[i, 0]
    for k in CORE:
        s = at_base[(at_base["mode"] == mode) & (at_base.method == k)].sort_values("REB")
        ax.plot(range(3), s["Vol %"], marker="o", lw=2.2, color=color_of(k), label=k)
        for x, v in enumerate(s["Vol %"]): ax.annotate(f"{v:.2f}", (x, v), textcoords="offset points", xytext=(0, 7 if k != "EW-RIE*" else -13), ha="center", fontsize=8, color=color_of(k))
    ax.set(title=f"{VN[mode]}: Vol theo tần suất tái cân bằng", xticks=range(3), xticklabels=[f"REB = {r}" for r in REBS], ylabel="Vol %/năm (gộp)")
    ax.margins(x=.12, y=.2)
    if i == 0: ax.legend(loc="upper left")
    vlim = np.abs(T2A.loc[T2A["mode"] == mode, "Sharpe sau phí"]).max()            # cùng thang màu cho 3 phương pháp trong một hàng
    for j, k in enumerate(CORE):
        M = T2A[(T2A["mode"] == mode) & (T2A.method == k)].pivot(index="REB", columns="COST", values="Sharpe sau phí")
        ax = axes[i, j + 1]
        im = ax.imshow(M.values, cmap="RdYlGn", norm=TwoSlopeNorm(0, -vlim, vlim), aspect="auto"); ax.grid(False)
        ax.set_xticks(range(3), [f"{c:.2%}" for c in M.columns]); ax.set_yticks(range(3), [f"REB {r}" for r in M.index] if j == 0 else [])
        for (y, x), v in np.ndenumerate(M.values): ax.text(x, y, f"{v:.2f}", ha="center", va="center", fontsize=10)
        ax.set_title(f"{k}", color=color_of(k)); ax.set_xlabel("COST một chiều")
    fig.colorbar(im, ax=axes[i, 1:], shrink=.85, label="Sharpe sau phí")
fig.suptitle("T2a – Vol theo REB (cột trái) và Sharpe sau phí theo REB × COST (xanh = dương, đỏ = âm)", x=.01, ha="left", fontweight="bold")
plt.show()

<!--CMT:t2a-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T2b. Phiên lợi suất 0 (mã đứng giá). So 3 mức lọc, chỉ dùng cửa sổ QUÁ KHỨ để loại mã:
#      không lọc | loại mã có > 20% phiên r = 0 (chuẩn) | loại mã có > 10% phiên r = 0.
VARS = {"không lọc": 1.0, "lọc > 20% (chuẩn)": 0.20, "lọc > 10%": 0.10}
T2B_RUN = {(v, mode): run(EST_CORE, dict(CONFIG, ZERO_MAX=z), lo) for v, z in VARS.items() for mode, lo in MODES.items()}
COMMON = sorted(set.intersection(*[set(d["out"]["NL-LW"]["r"].index) for d in T2B_RUN.values()]))   # kỳ thiếu mã bị bỏ ⇒ so trên ngày chung

def zero_share_in_universe(ulog):
    """Tỷ lệ r = 0 trung bình của các mã được chọn (đúng dữ liệu đưa vào bộ ước lượng)."""
    vals = []
    for u in ulog:
        s = POS[u["date"]]; win = R_BT.iloc[s - CONFIG["WINDOW"]:s][u["tickers"]].fillna(0)
        vals.append((win == 0).values.mean())
    return 100 * np.mean(vals)

rows = []
for (v, mode), d in T2B_RUN.items():
    info = {"Số kỳ hợp lệ": len(d["ulog"]), "Mã đủ ĐK (TB)": np.mean([u["n_eligible"] for u in d["ulog"]]),
            "Mã đủ ĐK (min)": min(u["n_eligible"] for u in d["ulog"]), "% r=0 trong vũ trụ": zero_share_in_universe(d["ulog"])}
    rows += [{"variant": v, "mode": mode, "method": k, **metrics(o, dates=COMMON), **info} for k, o in d["out"].items()]
T2B = pd.DataFrame(rows); save(T2B, "T2b_zero_return")

vol_tab = T2B.pivot_table(index="variant", columns=["mode", "method"], values="Vol %").reindex(columns=CORE, level=1).loc[list(VARS)]
info_tab = T2B[T2B["mode"] == "long_only"].groupby("variant")[["Số kỳ hợp lệ", "Mã đủ ĐK (TB)", "Mã đủ ĐK (min)", "% r=0 trong vũ trụ"]].first().loc[list(VARS)]
print(f"Bảng T2b – Vol %/năm trên {len(COMMON)} ngày OOS chung ({COMMON[0].date()} → {COMMON[-1].date()}); tổng số kỳ chuẩn: {len(U0)}")
show(pd.concat([vol_tab.rename(columns=VN, level=0), pd.concat({"Vũ trụ": info_tab}, axis=1)], axis=1), 2)

# hình: nhóm cột = mức lọc, cột = phương pháp (trục bắt đầu từ 0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, mode in zip(axes, MODES):
    for j, k in enumerate(CORE):
        vals = [vol_tab[(mode, k)].loc[v] for v in VARS]
        bars = ax.bar(np.arange(3) + (j - 1) * .26, vals, .25, color=color_of(k), label=k)
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set(title=f"{VN[mode]}: Vol theo mức lọc mã đứng giá", xticks=range(3), xticklabels=[f"{v}\n({int(info_tab.loc[v, 'Số kỳ hợp lệ'])}/{len(U0)} kỳ)" for v in VARS], ylim=(0, vol_tab.values.max() * 1.28))
axes[0].set_ylabel("Vol %/năm"); axes[0].legend(loc="upper left", ncol=3)
fig.tight_layout(); plt.show()

<!--CMT:t2b-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T2c. Quy ước chi phí. Trong run_backtest, phí kỳ = COST × Σ|w_mới − w_cũ|, tức COST là phí MỘT CHIỀU áp cho cả mua lẫn bán.
# HOSE thực tế: phí môi giới ≈ 0,15% mỗi chiều + thuế bán 0,1% ⇒ bất đối xứng (mua 0,15%, bán 0,25%).
def trade_cost(w_new, w_old, c_buy, c_sell):
    """Phí của một lần tái cân bằng khi mua/bán có mức phí khác nhau."""
    u = w_new.index.union(w_old.index); d = w_new.reindex(u, fill_value=0) - w_old.reindex(u, fill_value=0)
    return c_buy * d.clip(lower=0).sum() + c_sell * (-d).clip(lower=0).sum()

rng = np.random.default_rng(SEED)
for _ in range(200):          # unit test: có mã ra/vào vũ trụ và tỷ trọng âm; Σw = 1 ở cả hai kỳ
    a_ = pd.Series(rng.normal(size=6), index=list("ABCDEF")); a_ /= a_.sum()
    b_ = pd.Series(rng.dirichlet(np.ones(4)), index=list("DEFG"))
    u = a_.index.union(b_.index); turn = (a_.reindex(u, fill_value=0) - b_.reindex(u, fill_value=0)).abs().sum()
    assert np.isclose(trade_cost(a_, b_, .0025, .0025), .0025 * turn) and np.isclose(trade_cost(a_, b_, .0015, .0025), .002 * turn)
o_new = run({"NL-LW": est_nllw}, dict(CONFIG, REB=63, COST=0.004))["out"]["NL-LW"]           # chạy lại thật với COST khác
assert np.allclose(net(T2[63, "long_only"]["out"]["NL-LW"], 0.004).values, o_new["r_net"].values, atol=1e-14)
print("Unit test 200/200: Σw = 1 ⇒ giá trị mua = giá trị bán = turnover/2 ⇒ phí (mua 0,15%, bán 0,25%) ≡ COST đối xứng 0,20%.")
print("Kiểm chứng: r_net tính lại từ turnover trùng với chạy lại backtest thật (REB = 63, COST = 0,40%).")

SCEN = {"Gộp (chưa phí)": None, "COST 0,25% (gốc)": .0025, "Mua 0,15% / bán 0,25%": .002}
T2C = (pd.DataFrame([{"mode": VN[m], "method": k, "kịch bản": sc, "Sharpe": ann(o["r"] if c is None else net(o, c))[1]}
                     for m in MODES for k, o in T2[21, m]["out"].items() for sc, c in SCEN.items()])
         .pivot(index="method", columns=["mode", "kịch bản"], values="Sharpe")
         .reindex(index=CORE, columns=pd.MultiIndex.from_product([VN.values(), SCEN])))
save(T2C, "T2c_cost_convention"); print("Sharpe (REB = 21):"); show(T2C)

<!--CMT:t2c-->
(nhận xét điền sau khi xem hình)

### T3 – Đối chứng & kiểm định
**Câu hỏi:** EW-RIE\* có vượt các đối chứng mạnh trong cùng điều kiện, và chênh lệch có vượt dao động ngẫu nhiên của mẫu?
- **T3a** đối chứng mới: 1/N (chiến lược phân bổ, không phải bộ ước lượng), PCA k nhân tố (k ∈ {1, 3, 5}; PCA-auto chọn k chỉ trong cửa sổ huấn luyện), DCC-GARCH(1,1) (GARCH đơn biến bằng `arch` + DCC đầy đủ N×N).
- **T3b** bảng so sánh: Vol, Sharpe trước/sau phí, turnover, số mã hiệu dụng, thời gian chạy; Vol theo giai đoạn.
- **T3c** kiểm định phương sai Ledoit–Wolf (2011) với HAC và bootstrap khối; Sharpe Jobson–Korkie–Memmel; hiệu chỉnh Holm cho 5 so sánh mỗi chế độ.
- **T3d** hình: Vol kèm CI 95% và chênh lệch Vol kèm p Holm.

In [ ]:
# ---- T3: Đối chứng & kiểm định ----
def est_equal_weight_proxy(X):
    # 1/N: dùng ma trận đơn vị -> GMV = 1/N
    return np.eye(X.shape[1])
# TODO: mô hình nhân tố (PCA k nhân tố), DCC-GARCH (gói `arch`/`mgarch`),
# kiểm định Ledoit & Wolf (2011) "Robust performances hypothesis testing with the variance" (HAC + bootstrap),
# kiểm định Sharpe Jobson–Korkie với hiệu chỉnh Memmel (2003).

In [ ]:
# T3a. Đối chứng mới: 1/N, mô hình nhân tố PCA (k ∈ {1,3,5} và tự chọn k trong cửa sổ huấn luyện), DCC-GARCH(1,1).
from arch import arch_model
from scipy.optimize import minimize

def est_equal_weight_proxy(X):
    # 1/N: dùng ma trận đơn vị -> GMV = 1/N (chiến lược phân bổ, KHÔNG phải bộ ước lượng hiệp phương sai)
    return np.eye(X.shape[1])

def est_pca(k):
    # Σ = B Bᵀ + diag(ψ): k thành phần chính của hiệp phương sai mẫu, phần dư chéo (đối xứng, PSD theo cấu trúc)
    def est(X):
        S = np.cov(X, rowvar=False); lam, Q = np.linalg.eigh(S); B = Q[:, -k:] * np.sqrt(lam[-k:])
        return B @ B.T + np.diag(np.maximum(np.diag(S) - (B**2).sum(1), 1e-10))
    return est

PCA_K = []
def est_pca_auto(X, ks=(1, 3, 5), split=0.7):
    # chọn k CHỈ trong cửa sổ huấn luyện: ước lượng trên 70% đầu, đo phương sai GMV trên 30% cuối, rồi ước lượng lại trên cả cửa sổ
    n = int(len(X) * split)
    k = min(ks, key=lambda k: np.var(X[n:] @ gmv_unconstrained(est_pca(k)(X[:n]))))
    PCA_K.append(k); return est_pca(k)(X)

DCC_LOG, _DCC_CACHE = [], {}
def _dcc_ncl(p, Z, Qbar):
    # −composite log-lik (Engle, Shephard & Sheppard 2008): tổng log-lik 2×2 của các cặp kề nhau (i, i+1).
    # Full QML với N=100 vừa chậm vừa kéo a → 0 (thiên lệch đã biết khi N lớn) nên dùng CL.
    a, b = p
    if a < 0 or b < 0 or a + b >= 0.999: return 1e10
    z1, z2 = Z[:, :-1], Z[:, 1:]; q11, q22, q12 = np.diag(Qbar)[:-1], np.diag(Qbar)[1:], np.diag(Qbar, 1)
    c11, c22, c12 = q11.copy(), q22.copy(), q12.copy(); nll = 0.0
    for t in range(len(Z)):
        if t:
            c11 = (1-a-b)*q11 + a*z1[t-1]**2 + b*c11; c22 = (1-a-b)*q22 + a*z2[t-1]**2 + b*c22
            c12 = (1-a-b)*q12 + a*z1[t-1]*z2[t-1] + b*c12
        rho = c12 / np.sqrt(c11 * c22); d = 1 - rho**2
        nll += 0.5 * np.sum(np.log(d) + (z1[t]**2 + z2[t]**2 - 2*rho*z1[t]*z2[t]) / d)
    return nll

def est_dcc(X, h=21):
    # DCC-GARCH(1,1) (Engle 2002): bước 1 GARCH(1,1) đơn biến (`arch`) cho từng mã; bước 2 (a, b) bằng composite
    # likelihood + correlation targeting; ma trận Q_t đầy đủ N×N. Dự báo: D = căn phương sai GARCH trung bình h phiên tới, R = R_{T+1|T}; Σ = D R D.
    key = hash(X.tobytes())
    if key in _DCC_CACHE: return _DCC_CACHE[key]               # cùng cửa sổ ở long_only/short_ok → không ước lượng lại
    T, N = X.shape; Z = np.empty_like(X); hv = np.empty(N); bad = 0
    for i in range(N):
        try:
            r = arch_model(100 * X[:, i], mean="Constant", vol="GARCH", p=1, q=1, rescale=False).fit(disp="off", show_warning=False)
            ok = r.convergence_flag == 0 and r.params["alpha[1]"] + r.params["beta[1]"] < 1
        except Exception:
            ok = False
        if ok:
            Z[:, i] = r.std_resid; hv[i] = r.forecast(horizon=h, reindex=False).variance.values[-1].mean() / 1e4
        else:                                                   # dự phòng: phương sai mẫu (đếm vào DCC_LOG)
            bad += 1; Z[:, i] = (X[:, i] - X[:, i].mean()) / X[:, i].std(); hv[i] = X[:, i].var()
    Qbar = Z.T @ Z / T
    res = minimize(_dcc_ncl, x0=[0.02, 0.95], args=(Z, Qbar), method="Nelder-Mead", options=dict(xatol=1e-4, fatol=1e-3, maxiter=200))
    a, b = res.x if res.fun < 1e9 else (0.0, 0.0)
    Qt = Qbar.copy()
    for t in range(1, T + 1): Qt = (1 - a - b) * Qbar + a * np.outer(Z[t-1], Z[t-1]) + b * Qt    # Q_{T+1}
    d = 1 / np.sqrt(np.diag(Qt)); sd = np.sqrt(hv)
    S = (Qt * np.outer(d, d)) * np.outer(sd, sd)
    DCC_LOG.append(dict(a=a, b=b, garch_fail=bad, dcc_ok=bool(res.success))); _DCC_CACHE[key] = S
    return S

In [ ]:
# T3b. Bảng so sánh: cùng ngày ngoài mẫu, cùng vũ trụ, REB = 21, COST = 0,25%, hai chế độ.
T3_EST = {"1/N": est_equal_weight_proxy,
          **{f"PCA-k{k}": est_pca(k) for k in (1, 3, 5)}, "PCA-auto": est_pca_auto,
          "DCC-GARCH(1,1)": est_dcc, "EW-WeSpeR": ESTIMATORS["EW-WeSpeR"], **EST_CORE}
T3R = {mode: run(T3_EST, CONFIG, lo) for mode, lo in MODES.items()}

T3B = pd.DataFrame([{"mode": mode, "method": k, **metrics(o), "Tỷ trọng max %": o["m"]["Tỷ trọng max %"],
                     "Giây/lần ước lượng": T3R["long_only"]["sec"][k]}
                    for mode, d in T3R.items() for k, o in d["out"].items()])
save(T3B, "T3b_benchmark")
for mode in MODES:                                                     # kiểm tra hợp lý: 1/N ít giao dịch nhất
    assert T3B[T3B["mode"] == mode].set_index("method")["Turnover/kỳ"].idxmin() == "1/N"

tab = T3B.pivot(index="method", columns="mode", values=["Vol %", "Sharpe", "Sharpe sau phí", "Turnover/kỳ", "Số mã hiệu dụng"])
tab.columns = tab.columns.swaplevel(); tab = tab[[c for m in MODES for c in tab.columns if c[0] == m]].sort_values(("long_only", "Vol %"))
tab["Giây/lần ước lượng"] = T3B.drop_duplicates("method").set_index("method")["Giây/lần ước lượng"]
print("Bảng T3b – xếp theo Vol chỉ mua (Vol %/năm gộp; Sharpe sau phí trừ COST × turnover)"); show(tab.rename(columns=VN, level=0))

by_period = pd.concat({VN[m]: summarize(d["out"], [])[1] for m, d in T3R.items()}); save(by_period, "T3b_periods")
print("Vol %/năm theo giai đoạn thị trường:"); show(by_period, 2)

# nhật ký DCC / PCA-auto (chỉ có khi vừa tính mới, không dùng cache)
if len(DCC_LOG) >= len(U0):
    pd.DataFrame(DCC_LOG[:len(U0)]).assign(pca_auto_k=PCA_K[:len(U0)]).to_csv(f"{OUT_DIR}/T3_dcc_pca_log.csv", index=False)
dl_ = pd.read_csv(f"{OUT_DIR}/T3_dcc_pca_log.csv")
print(f"DCC-GARCH: a TB {dl_.a.mean():.3f}, b TB {dl_.b.mean():.3f}; {int(dl_.garch_fail.sum())}/{len(dl_) * CONFIG['N']} lần fit GARCH không hội tụ (dùng phương sai mẫu) | "
      f"PCA-auto chọn k = 1/3/5: {', '.join(str(int((dl_.pca_auto_k == k).sum())) for k in (1, 3, 5))} kỳ")

<!--CMT:t3b-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T3c. Kiểm định chênh lệch phương sai Ledoit–Wolf (2011) và Sharpe Jobson–Korkie–Memmel (JKM).
# Δ = log σ²(EW-RIE*) − log σ²(đối chứng) < 0 ⇒ EW-RIE* rủi ro thấp hơn. p-value: HAC và studentized circular block bootstrap (block 10/21/42); Holm cho 5 so sánh/chế độ.
def hac(u, L=None):
    """Ước lượng HAC Newey–West (Bartlett), L = ⌊4(T/100)^(2/9)⌋."""
    T = len(u); L = L or int(4 * (T / 100) ** (2 / 9)); S = u.T @ u / T
    for l in range(1, L + 1):
        G = u[l:].T @ u[:-l] / T; S += (1 - l / (L + 1)) * (G + G.T)
    return S

def _lw_stat(y, Psi=None, b=None):
    """Δ = log σ²_a − log σ²_b và sai số chuẩn (delta method); y_t = (r_a, r_b, r_a², r_b²)."""
    T = len(y); m = y.mean(0); va, vb = m[2] - m[0]**2, m[3] - m[1]**2
    g = np.array([-2 * m[0] / va, 2 * m[1] / vb, 1 / va, -1 / vb])
    if Psi is None:                                           # Ψ của bootstrap theo khối (LW 2008, công thức (9))
        z = (y - m)[:T // b * b].reshape(-1, b, 4).sum(1) / np.sqrt(b); Psi = z.T @ z / len(z)
    return np.log(va) - np.log(vb), np.sqrt(g @ Psi @ g / T)

def lw_var_test(ra, rb, block=21, M=2000, seed=SEED):
    """Trả về (Δ, sai số chuẩn, p HAC, p bootstrap khối vòng studentized)."""
    y = np.column_stack([ra, rb, ra**2, rb**2]); T = len(y)
    d, se = _lw_stat(y, hac(y - y.mean(0))); p_hac = 2 * norm.sf(abs(d / se))
    rng = np.random.default_rng(seed); l = T // block; stats = np.empty(M)
    for i in range(M):
        ix = ((rng.integers(0, T, l)[:, None] + np.arange(block)) % T).ravel()
        ds, ses = _lw_stat(y[ix], b=block); stats[i] = abs(ds - d) / ses
    return d, se, p_hac, (np.sum(stats >= abs(d / se)) + 1) / (M + 1)

def jkm_test(ra, rb):
    """Jobson–Korkie (1981) với hiệu chỉnh Memmel (2003); giả định iid chuẩn."""
    T = len(ra); m1, m2 = ra.mean(), rb.mean(); s1, s2 = ra.std(ddof=1), rb.std(ddof=1); s12 = np.cov(ra, rb)[0, 1]
    th = (2*s1**2*s2**2 - 2*s1*s2*s12 + .5*m1**2*s2**2 + .5*m2**2*s1**2 - m1*m2/(s1*s2)*s12**2) / T
    z = (s2 * m1 - s1 * m2) / np.sqrt(th); return z, 2 * norm.sf(abs(z))

# unit test nhanh: gần trùng ⇒ p lớn; gấp đôi biến động ⇒ p ≈ 0; JKM phản đối xứng
_r = np.random.default_rng(SEED).standard_normal(1500) * .01; _q = np.random.default_rng(2).standard_normal(1500) * .01
assert lw_var_test(_r, _r + 1e-5 * np.random.default_rng(1).standard_normal(1500), M=200)[3] > .1
assert lw_var_test(_r, 2 * _r[::-1], M=200)[2] < 1e-6
assert np.isclose(jkm_test(_r, _q)[0], -jkm_test(_q, _r)[0]) and jkm_test(_r + .003, _q - .003)[1] < 1e-6

BASELINES = ("NL-LW", "EW-NL-LW", "EW-WeSpeR", "1/N", "PCA-auto")           # đối chứng định trước, không chọn hậu kiểm
rows = []
for mode in MODES:
    out = T3R[mode]["out"]
    for b_ in BASELINES:
        ra, rb = out["EW-RIE*"]["r"].values, out[b_]["r"].values
        na, nb = (net(out[k], CONFIG["COST"]).values for k in ("EW-RIE*", b_))
        row = {"mode": mode, "pair": f"EW-RIE* vs {b_}", "ΔVol (pp)": (ra.std() - rb.std()) * np.sqrt(252) * 100}
        for blk in (10, 21, 42):
            d, se, p_hac, p_bt = lw_var_test(ra, rb, block=blk); lo_, hi_ = block_bootstrap_ci(ra, rb, block=blk)
            row.update({"Δlog σ²": d, "p LW-HAC": p_hac, f"p LW-boot b={blk}": p_bt, f"CI lo b={blk}": lo_, f"CI hi b={blk}": hi_})
        row.update({"p JKM gộp": jkm_test(ra, rb)[1], "p JKM sau phí": jkm_test(na, nb)[1]})
        rows.append(row)
T3C = pd.DataFrame(rows)

def holm(p):
    """Hiệu chỉnh Holm–Bonferroni; trả về p đã hiệu chỉnh theo thứ tự ban đầu."""
    p = np.asarray(p); order = np.argsort(p); adj = np.minimum(1, np.maximum.accumulate(p[order] * (len(p) - np.arange(len(p)))))
    out = np.empty_like(adj); out[order] = adj; return out
T3C["p Holm (b=21)"] = T3C.groupby("mode")["p LW-boot b=21"].transform(lambda s: pd.Series(holm(s.values), index=s.index))
save(T3C, "T3c_tests")

cols = ["ΔVol (pp)", "CI lo b=21", "CI hi b=21", "p LW-HAC", "p LW-boot b=10", "p LW-boot b=21", "p LW-boot b=42", "p Holm (b=21)", "p JKM gộp", "p JKM sau phí"]
print("Bảng T3c – EW-RIE* so với đối chứng (ΔVol < 0: EW-RIE* rủi ro thấp hơn; CI = bootstrap khối b = 21)")
show(T3C.assign(mode=T3C["mode"].map(VN)).set_index(["mode", "pair"])[cols], 4)

<!--CMT:t3c-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T3d. Hình: (trên) Vol từng phương pháp kèm CI 95%; (dưới) chênh lệch Vol của EW-RIE* so với từng đối chứng kèm CI và p Holm.
fig, axes = plt.subplots(2, 2, figsize=(14, 9), gridspec_kw=dict(height_ratios=[1.35, 1]))
for j, mode in enumerate(MODES):
    out = T3R[mode]["out"]; vol = pd.Series({k: o["m"]["Vol %"] for k, o in out.items()}).sort_values(ascending=False)
    ci = {k: block_bootstrap_ci(out[k]["r"].values, np.zeros(len(out[k]["r"]))) for k in vol.index}         # CI 95% của Vol từng chuỗi
    ax = axes[0, j]; y = np.arange(len(vol))
    ax.barh(y, vol.values, color=[color_of(k) for k in vol.index], height=.62)
    ax.errorbar(vol.values, y, xerr=[[max(vol[k] - ci[k][0], 0) for k in vol.index], [max(ci[k][1] - vol[k], 0) for k in vol.index]], fmt="none", ecolor="#52514e", lw=1, capsize=3)
    for yi, k in zip(y, vol.index): ax.text(ci[k][1] + .25, yi, f"{vol[k]:.2f}", va="center", fontsize=9)
    ax.set(title=f"{VN[mode]}: rủi ro ngoài mẫu (Vol %/năm) và CI 95%", yticks=y, yticklabels=vol.index, xlim=(0, vol.max() * 1.18)); ax.grid(axis="y", visible=False)

    t = T3C[T3C["mode"] == mode].iloc[::-1]; ax = axes[1, j]; y = np.arange(len(t))
    near = ~t["pair"].str.endswith("1/N")                                                      # 1/N cách xa các đối chứng khác ⇒ vẽ ngoài khung
    x_lo, x_hi = min(t.loc[near, "CI lo b=21"].min(), -.2) - .35, max(t.loc[near, "CI hi b=21"].max(), .2) + .35
    for yi, (pair, d_, lo_, hi_, p_) in enumerate(zip(t["pair"], t["ΔVol (pp)"], t["CI lo b=21"], t["CI hi b=21"], t["p Holm (b=21)"])):
        sig = p_ < .05; face = "#1baf7a" if sig else "white"
        if pair.endswith("1/N"):
            ax.scatter([x_lo + .08], [yi], marker="<", s=90, facecolor=face, edgecolor="#52514e", zorder=3)
            ax.text(x_lo + .22, yi, f"ΔVol = {d_:.1f} pp (ngoài khung), p = {pv(p_)}", va="center", fontsize=8)
        else:
            ax.plot([lo_, hi_], [yi, yi], color=GREY, lw=2.2, zorder=1)
            ax.scatter([d_], [yi], s=70, facecolor=face, edgecolor="#52514e", zorder=3)
            ax.annotate(f"p = {pv(p_)}", (d_, yi), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)
    ax.axvline(0, color="#52514e", lw=1); ax.set_xlim(x_lo, x_hi)
    ax.set(title=f"{VN[mode]}: ΔVol = EW-RIE* − đối chứng (pp; < 0 = EW-RIE* tốt hơn)", yticks=y, yticklabels=[p_.replace("EW-RIE* vs ", "") for p_ in t["pair"]]); ax.grid(axis="y", visible=False)
    if j == 0:
        ax.scatter([], [], s=70, color="#1baf7a", label="p Holm < 0,05"); ax.scatter([], [], s=70, facecolor="white", edgecolor="#52514e", label="không có ý nghĩa"); ax.legend(loc="lower right")
fig.tight_layout(); plt.show()

<!--CMT:t3d-->
(nhận xét điền sau khi xem hình)

### T5 – Vì sao VS giúp trên HOSE?
**Câu hỏi:** chuẩn hóa biến động (VS) có giảm rủi ro không, khi nào, và nhờ cơ chế nào?
- **T5a** tách tác động chỉ do VS (RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW) và phân rã theo giai đoạn thị trường.
- **T5b** dự báo biến động 21 phiên: EWMA vs cửa sổ phẳng (RMSE, QLIKE).
- **T5c** phổ trị riêng trước/sau VS và condition number của Σ đưa vào GMV.

In [ ]:
# ---- T5: Vì sao VS giúp trên HOSE? ----
# Gợi ý: (a) so sánh VS chỉ chuẩn hóa biến động riêng lẻ vs. cả nhân tố thị trường;
# (b) đo độ chính xác dự báo biến động 21 phiên của EWMA vs. cửa sổ phẳng trên HOSE;
# (c) phân rã lợi ích VS theo giai đoạn (bảng RESULTS[mode]["periods"]).

In [ ]:
# T5a. Tách tác động của VS: mỗi cặp chỉ khác bước chuẩn hóa biến động (RIE ↔ VS-RIE, NL-LW ↔ VS-NL-LW); dùng lại lần chạy chính §4.
ABL = [("RIE", "VS-RIE"), ("NL-LW", "VS-NL-LW")]
rows, per = [], []
for mode in MODES:
    out = RESULTS[mode]["out"]
    for base, vs in ABL:
        lo_, hi_ = block_bootstrap_ci(out[vs]["r"].values, out[base]["r"].values)                 # CI 95% của Vol(VS) − Vol(gốc)
        mb, mv = metrics(out[base]), metrics(out[vs])
        rows.append({"Chế độ": VN[mode], "Cặp": f"{vs} vs {base}", "Vol gốc": mb["Vol %"], "Vol VS": mv["Vol %"], "ΔVol (pp)": mv["Vol %"] - mb["Vol %"],
                     "CI lo": lo_, "CI hi": hi_, "Sharpe sau phí gốc": mb["Sharpe sau phí"], "Sharpe sau phí VS": mv["Sharpe sau phí"],
                     "Turnover gốc": mb["Turnover/kỳ"], "Turnover VS": mv["Turnover/kỳ"]})
        per.append((RESULTS[mode]["periods"].loc[vs] - RESULTS[mode]["periods"].loc[base]).rename((VN[mode], f"{vs} − {base}")))
T5A = pd.DataFrame(rows).set_index(["Chế độ", "Cặp"]); T5P = pd.DataFrame(per).T
save(T5A, "T5a_vs_ablation"); save(T5P, "T5a_vs_ablation_periods")
print("Bảng T5a – tác động của VS (Vol %/năm gộp; CI 95% bootstrap khối của ΔVol)"); show(T5A)
print("ΔVol (pp) do VS theo giai đoạn thị trường (< 0: VS giảm rủi ro):"); show(T5P, 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.3), sharey=True)
for ax, mode in zip(axes, VN.values()):
    d = T5P[mode]; x = np.arange(len(d))
    for j, (c, col) in enumerate(zip(d.columns, ["#2a78d6", "#eb6834"])):
        bars = ax.bar(x + (j - .5) * .38, d[c], .36, color=col, label=c); ax.bar_label(bars, fmt="%.1f", fontsize=7, padding=2)
    ax.axhline(0, color="#52514e", lw=1)
    ax.set_xticks(x, [i.replace(" ", "\n", 1) for i in d.index], fontsize=8); ax.set_title(f"{mode}: ΔVol do VS theo giai đoạn (pp/năm; < 0 ⇒ VS giảm rủi ro)")
axes[0].legend(loc="lower right"); axes[0].set_ylabel("ΔVol (pp/năm)")
fig.tight_layout(); plt.show()

<!--CMT:t5a-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T5b. Dự báo phương sai 21 phiên tới: EWMA (hl ∈ {10, 20, 40}, đúng hàm ewma_standardize mà VS dùng) vs cửa sổ phẳng 63/250 phiên.
# Mẫu: các mã trong vũ trụ chuẩn mỗi kỳ; dự báo chỉ dùng cửa sổ quá khứ. Mục tiêu RV = trung bình r² của 21 phiên sau.
# Sai số: RMSE của vol năm hóa (%) và QLIKE = log h + RV/h (Patton 2011; xác định cả khi RV = 0).
W, H = CONFIG["WINDOW"], 21; rows = []
for u in U0:
    s = POS[u["date"]]
    if s + H > len(R_BT): continue
    X = R_BT.iloc[s - W:s][u["tickers"]].fillna(0).values                               # cửa sổ quá khứ
    rv = (R_BT.iloc[s:s + H][u["tickers"]].fillna(0).values ** 2).mean(0)               # biến động thực hiện 21 phiên sau
    forecasts = {**{f"EWMA hl={h}": ewma_standardize(X, h)[1] ** 2 for h in (10, 20, 40)},
                 **{f"Phẳng {w} phiên": (X[-w:] ** 2).mean(0) for w in (63, W)}}
    period = next((p for p, a_, b_ in PERIODS if pd.Timestamp(a_) <= u["date"] <= pd.Timestamp(b_)), None)
    for name, h_ in forecasts.items():
        h_ = np.maximum(h_, 1e-12)
        rows.append({"period": period, "model": name, "se": np.mean((np.sqrt(h_) - np.sqrt(rv)) ** 2), "ql": np.mean(np.log(h_) + rv / h_)})
F = pd.DataFrame(rows); F = pd.concat([F, F.assign(period="Toàn mẫu")])
T5B = F.groupby(["period", "model"], sort=False).agg(RMSE=("se", lambda x: np.sqrt(x.mean() * 252) * 100), QLIKE=("ql", "mean"), n=("se", "size"))
save(T5B, "T5b_vol_forecast")

models = ["EWMA hl=10", "EWMA hl=20", "EWMA hl=40", "Phẳng 63 phiên", f"Phẳng {W} phiên"]
tbl = T5B["RMSE"].unstack()[models]; tbl.loc["QLIKE (toàn mẫu)"] = T5B.xs("Toàn mẫu")["QLIKE"]
print("Bảng T5b – RMSE dự báo vol 21 phiên (%/năm) theo giai đoạn; dòng cuối: QLIKE toàn mẫu (thấp hơn = tốt hơn)"); show(tbl)

M = tbl.drop(index="QLIKE (toàn mẫu)"); x = np.arange(len(M)); shade = ["#2a78d6", "#1baf7a", "#7fbf9c", "#eda100", "#9aa0a6"]
fig, ax = plt.subplots(figsize=(13, 4.2))
for j, (c, col) in enumerate(zip(M.columns, shade)):
    ax.bar(x + (j - 2) * .16, M[c], .15, color=col, label=c)
best = M.idxmin(axis=1)
for i, (period, m_) in enumerate(best.items()): ax.plot(i + (list(M.columns).index(m_) - 2) * .16, M.loc[period, m_] + .6, marker="v", color="#52514e", ms=6)
ax.set_xticks(x, [i.replace(" ", "\n", 1) for i in M.index], fontsize=9); ax.set_ylabel("RMSE vol (%/năm)")
ax.set_title("T5b – sai số dự báo vol 21 phiên (▼ = phương pháp tốt nhất của giai đoạn)"); ax.legend(ncol=5, loc="upper left")
ax.set_ylim(0, M.values.max() * 1.15); fig.tight_layout(); plt.show()

<!--CMT:t5b-->
(nhận xét điền sau khi xem hình)

In [ ]:
# T5c. Phổ trị riêng trước/sau VS (một kỳ ví dụ) và condition number theo kỳ. "Trước": tương quan mẫu của X; "sau": của Z = X/σ_EWMA
# (bỏ HL_VOL phiên đầu, đúng make_vs). Thêm condition number của Σ THỰC SỰ đưa vào GMV.
hv_, rows = CONFIG["HL_VOL"], []
window_of = lambda u: R_BT.iloc[POS[u["date"]] - W:POS[u["date"]]][u["tickers"]].fillna(0).values
for u in U0:
    X = window_of(u); Z = ewma_standardize(X, hv_)[0][hv_:]
    ex, ez = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False)), np.linalg.eigvalsh(np.corrcoef(Z, rowvar=False))
    r = {"date": u["date"], "cond corr(X)": ex[-1] / ex[0], "cond corr(Z), VS": ez[-1] / ez[0], "λ1/N trước VS": ex[-1] / len(ex), "λ1/N sau VS": ez[-1] / len(ez)}
    for k in ("NL-LW", "VS-NL-LW", "RIE", "VS-RIE"):
        e = check_cov(ESTIMATORS[k](X), k); r[f"cond Σ {k}"] = e[-1] / e[0]
    rows.append(r)
T5C = pd.DataFrame(rows).set_index("date"); save(T5C, "T5c_condition_numbers")
print("Bảng T5c – condition number và trị riêng lớn nhất (trung bình / trung vị trên các kỳ)"); show(T5C.agg(["mean", "median"]).T, 2)

ex_u = next(u for u in U0 if u["date"] >= pd.Timestamp("2020-03-01"))                      # kỳ ví dụ: ngay sau COVID
X = window_of(ex_u); Z = ewma_standardize(X, hv_)[0][hv_:]
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.2))
for lab, M_, c in (("trước VS: corr(X)", X, GREY), ("sau VS: corr(Z)", Z, "#1baf7a")):
    ev = np.sort(np.linalg.eigvalsh(np.corrcoef(M_, rowvar=False)))[::-1]
    ax[0].plot(np.arange(1, len(ev) + 1), ev, color=c, lw=2.2, marker="o", ms=3, label=lab)
ax[0].axhline((1 + np.sqrt(CONFIG["N"] / W)) ** 2, ls=":", color="#52514e", label="biên MP λ₊ (nhiễu thuần)")
ax[0].set(yscale="log", xlabel="thứ hạng trị riêng", ylabel="trị riêng", title=f"Phổ tương quan, kỳ {ex_u['date'].date()} (N = {CONFIG['N']}, T = {W})"); ax[0].legend()
for k, c in (("cond Σ NL-LW", "#2a78d6"), ("cond Σ VS-NL-LW", "#eda100"), ("cond corr(X)", GREY), ("cond corr(Z), VS", "#1baf7a")):
    ax[1].plot(T5C.index, T5C[k], color=c, lw=2, label=k)
ax[1].set(yscale="log", title="Condition number theo kỳ tái cân bằng", ylabel="condition number"); ax[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

<!--CMT:t5c-->
(nhận xét điền sau khi xem hình)

### T6 – Khái quát: số tài sản N, cửa sổ T, half-life
**Câu hỏi:** EW-RIE\* có duy trì hiệu quả khi đổi quy mô và độ dài dữ liệu? Đổi từng yếu tố, giữ nguyên phần còn lại: N ∈ {50, 100, 150}, T ∈ {125, 250, 500}, h ∈ {30, 60, 120}. T_eff = 1/Σw², q_eff = N·Σw². VN30/HNX: chưa làm (cần lịch sử thành phần rổ).

In [ ]:
# ---- T6: Khái quát theo N, cửa sổ T, half-life (đổi TỪNG yếu tố, giữ nguyên phần còn lại) ----
# NL-LW, EW-NL-LW, EW-RIE*; cả hai chế độ. So các mức của MỘT yếu tố trên NGÀY OOS CHUNG (T = 500 bắt đầu muộn hơn).
GRID = [("N", v) for v in (50, 100, 150)] + [("WINDOW", v) for v in (125, 250, 500)] + [("HL_EW", v) for v in (30, 60, 120)]
SHORT = {"N": "N", "WINDOW": "T", "HL_EW": "h"}

T6R = {}
for factor, value in GRID:
    cfg = dict(CONFIG, **{factor: value}); est = {k: build_estimators(cfg)[k] for k in CORE}      # EW phụ thuộc HL_EW của cfg
    for mode, lo in MODES.items(): T6R[factor, value, mode] = run(est, cfg, lo)

rows = []
for factor in ("N", "WINDOW", "HL_EW"):
    values = [v for f, v in GRID if f == factor]
    for mode in MODES:
        common = sorted(set.intersection(*[set(T6R[factor, v, mode]["out"]["NL-LW"]["r"].index) for v in values]))
        for value in values:
            d = T6R[factor, value, mode]; o = d["out"]; cfg = dict(CONFIG, **{factor: value}); w = ew_weights(cfg["WINDOW"], cfg["HL_EW"])
            ra, rb = o["EW-RIE*"]["r"].loc[common].values, o["EW-NL-LW"]["r"].loc[common].values; lo_, hi_ = block_bootstrap_ci(ra, rb)
            rows.append({"factor": factor, "value": value, "mode": mode, "N": cfg["N"], "T": cfg["WINDOW"], "h": cfg["HL_EW"], "q": cfg["N"] / cfg["WINDOW"],
                         "T_eff": 1 / (w ** 2).sum(), "q_eff": cfg["N"] * (w ** 2).sum(), **{f"Vol {k}": ann(o[k]["r"].loc[common])[0] for k in CORE},
                         "Δ": (ra.std() - rb.std()) * np.sqrt(252) * 100, "CI lo": lo_, "CI hi": hi_, "Số ngày chung": len(common), "Số kỳ": len(d["ulog"]),
                         "giây/lần EW-RIE*": d["sec"].get("EW-RIE*", np.nan), "Newton fail": d["newton"][0], "Newton tổng": d["newton"][1]})
T6 = pd.DataFrame(rows); save(T6, "T6_generalization")
for (factor, mode), g in T6.groupby(["factor", "mode"]): assert g["Số ngày chung"].nunique() == 1        # cùng ngày chung trong mỗi yếu tố
T6["cấu hình"] = T6.apply(lambda r: f"{SHORT[r.factor]}={r.value}", axis=1)
T6["Δ (*)"] = T6.apply(lambda r: f2(r["Δ"], 3) + ("*" if r["CI hi"] < 0 or r["CI lo"] > 0 else ""), axis=1)       # * = CI 95% không chứa 0

tb = T6.pivot(index="cấu hình", columns="mode", values=["Vol NL-LW", "Vol EW-NL-LW", "Vol EW-RIE*", "Δ (*)"])
L6 = T6[T6["mode"] == "long_only"].set_index("cấu hình")                                                        # thứ tự N → T → h
tb.columns = tb.columns.swaplevel(); tb = tb[[c for m in MODES for c in tb.columns if c[0] == m]].reindex(L6.index)
tb[("", "q_eff")] = L6["q_eff"]; tb[("", "ngày chung")] = L6["Số ngày chung"]
print("Bảng T6 – Vol %/năm (ngày OOS chung trong mỗi yếu tố); Δ = Vol(EW-RIE*) − Vol(EW-NL-LW), pp; * = CI 95% bootstrap khối không chứa 0")
show(tb.rename(columns=VN, level=0), 2)
print("Thời gian và độ ổn định số của EW-RIE* (chỉ mua):"); show(L6[["Số kỳ", "giây/lần EW-RIE*", "Newton fail", "Newton tổng"]], 3)

# hình: hàng trên = Vol; hàng dưới = Δ kèm CI 95%; cột = N, T, q_eff (gộp mọi cấu hình), h
LOD = T6[T6["mode"] == "long_only"]
panels = [("N", "N", LOD[LOD.factor == "N"]), ("cửa sổ T", "T", LOD[LOD.factor == "WINDOW"]), ("q_eff = N·Σw² (gộp mọi cấu hình)", "q_eff", LOD), ("half-life h", "h", LOD[LOD.factor == "HL_EW"])]
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex="col")
for j, (title, xcol, d) in enumerate(panels):
    d = d.sort_values(xcol); is_scatter = xcol == "q_eff"
    for k in CORE: axes[0, j].plot(d[xcol], d[f"Vol {k}"], marker="o", ls="none" if is_scatter else "-", lw=2.2, ms=6, color=color_of(k), label=k)
    axes[1, j].errorbar(d[xcol], d["Δ"], yerr=[np.clip(d["Δ"] - d["CI lo"], 0, None), np.clip(d["CI hi"] - d["Δ"], 0, None)], fmt="o", ls="none" if is_scatter else "-",
                        color="#1baf7a", capsize=4, lw=2)
    axes[1, j].axhline(0, color="#52514e", lw=1); axes[0, j].set_title(f"Chỉ mua: Vol theo {title}", fontsize=10); axes[1, j].set_xlabel(title)
    if is_scatter: axes[0, j].axvline(1, color="#52514e", ls="--", lw=.8); axes[1, j].axvline(1, color="#52514e", ls="--", lw=.8)
axes[0, 0].set_ylabel("Vol %/năm"); axes[1, 0].set_ylabel("Δ = EW-RIE* − EW-NL-LW (pp)\nCI 95%"); axes[0, 0].legend(fontsize=8)
fig.suptitle("T6 – Vol (trên) và Δ Vol của EW-RIE* so với EW-NL-LW (dưới); đường đứt: q_eff = 1 ⇔ N = T_eff", x=.01, ha="left", fontweight="bold")
fig.tight_layout(); plt.show()

<!--CMT:t6-->
(nhận xét điền sau khi xem hình)

### Tự kiểm tra

In [ ]:
# ---- Tự kiểm tra T1–T6 ----
rng_ = np.random.default_rng(0); X_ = rng_.standard_normal((250, 100)); l_ = np.linalg.eigvalsh(np.corrcoef(X_, rowvar=False)); z_ = l_ - .1j
g_ = (1 / (z_[:, None] - l_[None, :])).mean(1); e_ = np.max(np.abs(_xi(l_, l_, np.full(250, 1 / 250), 100, .1) - l_ / np.abs(1 - .4 + .4 * z_ * g_) ** 2) / l_)
again = run_backtest(P, V, EST_CORE, dict(CONFIG, REB=63), True)                      # chạy lại không cache
checks = {
    "§1b Test 1 chạy lại (thư viện không đổi trạng thái)": e_ < 1e-10,
    "Không NaN/inf trong mọi CSV của results_v5/": all(np.isfinite(pd.read_csv(f"{OUT_DIR}/{f}").select_dtypes("number").values).all() for f in os.listdir(OUT_DIR) if f.endswith(".csv")),
    "run_backtest có lọc ≡ bản thầy khi ZERO_MAX = 1.0 (assert §3b)": True,
    "run() trùng §4 (assert §6.0)": True,
    "Common-sample + đối xứng/PSD/NaN của mọi Σ (assert trong run/timed)": True,
    "1/N có turnover thấp nhất": all(T3B[T3B["mode"] == m].set_index("method")["Turnover/kỳ"].idxmin() == "1/N" for m in MODES),
    "COST tăng ⇒ Sharpe sau phí không tăng (assert T2a)": True,
    "REB tăng ⇒ turnover/kỳ tăng, turnover/năm giảm": all(
        (T2A[(T2A.COST == CONFIG["COST"]) & (T2A["mode"] == m) & (T2A.method == k)].sort_values("REB")[c].diff().dropna() * s > 0).all()
        for m in MODES for k in CORE for c, s in (("Turnover/kỳ", 1), ("Turnover/năm", -1))),
    "Tái lập: chạy lại REB = 63 (không cache) trùng kết quả": all(np.array_equal(again[k]["r"].values, T2[63, "long_only"]["out"][k]["r"].values) for k in CORE),
    f"Newton không hội tụ (T6): {int(T6['Newton fail'].sum())} lần": T6["Newton fail"].sum() == 0,
    f"EW-WeSpeR dự phòng sang EW-RIE*: {_WSP_SAFE['fallback']}/{_WSP_SAFE['total']} lần": _WSP_SAFE["fallback"] == 0,
    "Lọc ZERO_MAX chỉ dùng cửa sổ quá khứ (mọi mã trong vũ trụ có r = 0 ≤ ZERO_MAX)": all(
        ((R_BT.iloc[POS[u["date"]] - CONFIG["WINDOW"]:POS[u["date"]]][u["tickers"]] == 0).mean() <= CONFIG["ZERO_MAX"]).all() for u in U0)}
for k, v in checks.items(): print(("✔ " if v else "✘ ") + k)
assert all(checks.values())

## Ghi chú
- **Khai báo dùng AI:** mã nguồn được xây dựng với hỗ trợ của trợ lý AI; nhóm cần rà soát, hiểu và khai báo theo quy định của tạp chí.
- **Tính mới:** công thức tổng quát cho nhiễu nhân có trong Bun–Bouchaud–Potters (2016, 2017). Cần T4 xác nhận phần cụ thể hóa cho EWMA, chọn nhánh, xử lý ngoại lai chưa được công bố.
- **Tái lập:** cố định `seed`; ghi lại phiên bản thư viện (ô đầu) vào phụ lục.

## Ghi chú v5
- **Mã hủy niêm yết:** lợi suất các ngày sau ngày giao dịch cuối = 0 (giữ tiền mặt đến kỳ tái cân bằng kế tiếp), đúng quy ước của thầy. 19 mã hủy không có giá điều chỉnh từ VCI (`missing_tickers.csv`).
- **EW-WeSpeR** là bản cài lại bằng numpy theo các phương trình của Oriol (2024); chưa đối chiếu với mã gốc của tác giả.
- **EW-NL-LW** dùng nhánh p > n của Ledoit–Wolf khi N > T_eff; T_eff = 1/Σw² không nguyên.
- **Để sau:** T4 (lý thuyết, LaTeX), T6 phần VN30/HNX, T7 (viết bài).
- **Chạy trên web (Colab):** chạy từ trên xuống; ô T1.0 tự tải `data_hose/` từ GitHub. Thời gian ≈ 1–1,5 giờ (DCC-GARCH ở T3 và lưới REB = 5 ở T2 chậm nhất).